# RAF-DB roll-only correction sweep (ADR-028)

Measures the decision taken after the 3DDFA-V2 benchmark: estimate the face **roll** with 3DDFA-V2 and undo it with a plain **2D rotation**; yaw/pitch are not corrected and nothing is warped.

For every RAF-DB test image (3,068) and every angle in `ANGLES` (default 0/15/30/45/60/90°): `raw` = rotated image → EMOCA → POSTER / MA3D-Net; `frontal` = rotated image → 3DDFA-V2 roll estimate → rotate by −roll → EMOCA → POSTER / MA3D-Net. When |roll| < `MIN_ROLL` the corrected image equals the input, so the raw result is reused. The summary then evaluates every threshold T in `ROLL_THRESHOLDS` post hoc (use the corrected image only when |roll| ≥ T) and recommends the T with the best mean accuracy over angles that loses ≤ 0.2 points on upright images.

Run with the existing private Kaggle Dataset attached, **GPU T4 x2**, Internet on. Progress while the full run is going: open the Output panel → `rafdb5/results/roll_sweep/full/progress_shard0.json` and `progress_shard1.json` (percent, seconds per row, ETA). The last cell zips the results to `/kaggle/working/roll_results.zip` for download. Numbers only; no images are saved.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

def run_quiet(command, *, label, env=None):
    completed = subprocess.run(
        command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, errors='replace',
    )
    if completed.returncode:
        tail = (completed.stdout or '').strip()[-4000:]
        raise RuntimeError(f'{label} failed (exit {completed.returncode}): {tail}')

# The project owner has already registered and accepted FLAME/EMOCA terms for this research run.
ACCEPT_FLAME_EMOCA_LICENSES = True
RUN_FULL_BENCHMARK = True
ANGLES = [0, 15, 30, 45, 60, 90]  # in-plane rotations; rows = 3068 x len(ANGLES)
MIN_ROLL = 5.0  # below this |roll| the corrected image equals the input (raw result reused)
ROLL_THRESHOLDS = [5, 10, 15, 20, 25, 30, 35, 40, 50, 60]  # evaluated post hoc
BUNDLE_ROOT_OVERRIDE = None  # Optional: exact path such as /kaggle/input/<dataset-slug>/bundle
WORKDIR = Path('/kaggle/working/rafdb5')

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
gpu_count = torch.cuda.device_count()
assert gpu_count == 2, f'This run is configured for T4 x2; found {gpu_count} visible GPU(s).'

if BUNDLE_ROOT_OVERRIDE:
    BUNDLE_ROOT = Path(BUNDLE_ROOT_OVERRIDE).resolve()
else:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('BUNDLE_MANIFEST.json')
                  if (p.parent / 'dataset' / 'test_labels.csv').is_file()]
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one attached Kaggle bundle; found {candidates}. Set BUNDLE_ROOT_OVERRIDE.')
    BUNDLE_ROOT = candidates[0]
KAGGLE_DIR = BUNDLE_ROOT.parent
SCRIPTS = KAGGLE_DIR / 'scripts'
assert (SCRIPTS / 'prepare_runtime.py').is_file(), f'Bundle scripts missing beside {BUNDLE_ROOT}'


In [ ]:
# Keep Kaggle's preinstalled NumPy and CUDA PyTorch; the bundle pin is for isolated local environments.
requirements = BUNDLE_ROOT / 'environment' / 'requirements-worker.txt'
runtime_requirements = WORKDIR / 'requirements-worker-kaggle.txt'
WORKDIR.mkdir(parents=True, exist_ok=True)
requirements_lines = requirements.read_text(encoding='utf-8').splitlines()
runtime_requirements.write_text(
    '\n'.join(line for line in requirements_lines
                if not line.strip().lower().startswith('numpy==')) + '\n',
    encoding='utf-8',
)
run_quiet([sys.executable, '-m', 'pip', 'install', '-r', str(runtime_requirements)],
          label='Worker package installation')
import numpy as np
import cv2
run_quiet([sys.executable, str(SCRIPTS / 'verify_bundle.py'), str(BUNDLE_ROOT)],
          label='Bundle verification')


In [ ]:
# Stage project code and make sure the optional M6 package is complete in older uploaded Datasets.
run_quiet([sys.executable, str(SCRIPTS / 'prepare_runtime.py'),
           '--bundle-root', str(BUNDLE_ROOT), '--workdir', str(WORKDIR)],
          label='Runtime staging')
import base64
m6_pipeline = WORKDIR / 'project' / 'services' / 'image_preprocessing' / 'pipeline.py'
m6_pipeline.write_bytes(base64.b64decode('ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRpbWUgaW1wb3J0IHBlcmZfY291bnRlcgpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBQcm90b2NvbAppbXBvcnQgbWF0aAppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBQSUwgaW1wb3J0IEltYWdlLCBJbWFnZU9wcwoKY2xhc3MgRGV0ZWN0b3IoUHJvdG9jb2wpOgogICAgbW9kZWxfaWQ6IHN0cgogICAgZGVmIGRldGVjdChzZWxmLCBpbWFnZTogSW1hZ2UuSW1hZ2UsIGNvbmZpZGVuY2U6IGZsb2F0KSAtPiBsaXN0W2RpY3Rbc3RyLCBBbnldXTogLi4uCmNsYXNzIFJlc3RvcmVyKFByb3RvY29sKToKICAgIG1vZGVsX2lkOiBzdHIKICAgIGRlZiByZXN0b3JlKHNlbGYsIGltYWdlOiBJbWFnZS5JbWFnZSwgZmlkZWxpdHk6IGZsb2F0IHwgTm9uZSA9IE5vbmUpIC0+IEltYWdlLkltYWdlOiAuLi4KCkBkYXRhY2xhc3MKY2xhc3MgUGlwZWxpbmVDb25maWc6CiAgICBjb25maWRlbmNlOiBmbG9hdCA9IDAuNQogICAgcGFkZGluZzogZmxvYXQgPSAwLjE1CiAgICBmaWRlbGl0eTogZmxvYXQgPSAwLjcKICAgIG1heF9zaWRlOiBpbnQgPSA0MDk2CiAgICBub19mYWNlX3BvbGljeTogc3RyID0gImtlZXAiCiAgICBudW1weV9jb2xvcjogc3RyID0gIlJHQiIKQGRhdGFjbGFzcwpjbGFzcyBGYWNlUmVzdWx0OgogICAgYmJveF94eXh5OiBsaXN0W2ludF0KICAgIGNyb3BfYmJveF94eXh5OiBsaXN0W2ludF0KICAgIGNvbmZpZGVuY2U6IGZsb2F0CiAgICBjcm9wOiBJbWFnZS5JbWFnZQpAZGF0YWNsYXNzCmNsYXNzIFBpcGVsaW5lUmVzdWx0OgogICAgZW5oYW5jZWQ6IEltYWdlLkltYWdlIHwgTm9uZQogICAgZmFjZXM6IGxpc3RbRmFjZVJlc3VsdF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGlucHV0X2RpbWVuc2lvbnM6IGxpc3RbaW50XSB8IE5vbmUgPSBOb25lCiAgICBvdXRwdXRfZGltZW5zaW9uczogbGlzdFtpbnRdIHwgTm9uZSA9IE5vbmUKICAgIG1vZGVsX2lkczogZGljdFtzdHIsIHN0cl0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIHN0YWdlX3RpbWVzX21zOiBkaWN0W3N0ciwgZmxvYXRdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCiAgICBzdGF0dXM6IHN0ciA9ICJlcnJvciIKICAgIGVycm9yOiBzdHIgfCBOb25lID0gTm9uZQoKZGVmIF9pbnB1dF9yZ2IoaW1hZ2U6IHN0ciB8IFBhdGggfCBJbWFnZS5JbWFnZSB8IG5wLm5kYXJyYXksIG51bXB5X2NvbG9yOiBzdHIpIC0+IEltYWdlLkltYWdlOgogICAgaWYgaXNpbnN0YW5jZShpbWFnZSwgKHN0ciwgUGF0aCkpOgogICAgICAgIHdpdGggSW1hZ2Uub3BlbihpbWFnZSkgYXMgc3JjOiByZXR1cm4gSW1hZ2VPcHMuZXhpZl90cmFuc3Bvc2Uoc3JjKS5jb252ZXJ0KCJSR0IiKS5jb3B5KCkKICAgIGlmIGlzaW5zdGFuY2UoaW1hZ2UsIEltYWdlLkltYWdlKTogcmV0dXJuIEltYWdlT3BzLmV4aWZfdHJhbnNwb3NlKGltYWdlKS5jb252ZXJ0KCJSR0IiKS5jb3B5KCkKICAgIGlmIGlzaW5zdGFuY2UoaW1hZ2UsIG5wLm5kYXJyYXkpOgogICAgICAgIGlmIGltYWdlLmR0eXBlICE9IG5wLnVpbnQ4IG9yIGltYWdlLm5kaW0gIT0gMyBvciBpbWFnZS5zaGFwZVsyXSAhPSAzIG9yIG1pbihpbWFnZS5zaGFwZVs6Ml0pIDwgMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTnVtUHkgaW5wdXQgbXVzdCBiZSBub24tZW1wdHkgSHhXeDMgdWludDgiKQogICAgICAgIGlmIG51bXB5X2NvbG9yIG5vdCBpbiAoIlJHQiIsICJCR1IiKTogcmFpc2UgVmFsdWVFcnJvcigibnVtcHlfY29sb3IgbXVzdCBiZSBSR0Igb3IgQkdSIikKICAgICAgICBhcnIgPSBpbWFnZSBpZiBudW1weV9jb2xvciA9PSAiUkdCIiBlbHNlIGltYWdlWy4uLiwgOjotMV0KICAgICAgICByZXR1cm4gSW1hZ2UuZnJvbWFycmF5KG5wLmFzY29udGlndW91c2FycmF5KGFyciksICJSR0IiKQogICAgcmFpc2UgVHlwZUVycm9yKCJpbnB1dCBtdXN0IGJlIGEgcGF0aCwgUElMIGltYWdlLCBvciBOdW1QeSBSR0IgYXJyYXkiKQoKZGVmIF94eXh5KGJveDogQW55LCB3aWR0aDogaW50LCBoZWlnaHQ6IGludCkgLT4gbGlzdFtpbnRdOgogICAgdiA9IG5wLmFzYXJyYXkoYm94LCBkdHlwZT1ucC5mbG9hdDY0KS5yZXNoYXBlKC0xKQogICAgaWYgbGVuKHYpICE9IDQgb3Igbm90IG5wLmlzZmluaXRlKHYpLmFsbCgpOiByYWlzZSBWYWx1ZUVycm9yKCJkZXRlY3RvciByZXR1cm5lZCBtYWxmb3JtZWQgYmJveF94eXh5IikKICAgIHgxLCB5MSwgeDIsIHkyID0gdgogICAgeDEsIHgyID0gc29ydGVkKChpbnQobnAuZmxvb3IoeDEpKSwgaW50KG5wLmNlaWwoeDIpKSkpCiAgICB5MSwgeTIgPSBzb3J0ZWQoKGludChucC5mbG9vcih5MSkpLCBpbnQobnAuY2VpbCh5MikpKSkKICAgIHgxLCB4MiA9IG1heCgwLCBtaW4od2lkdGgsIHgxKSksIG1heCgwLCBtaW4od2lkdGgsIHgyKSkKICAgIHkxLCB5MiA9IG1heCgwLCBtaW4oaGVpZ2h0LCB5MSkpLCBtYXgoMCwgbWluKGhlaWdodCwgeTIpKQogICAgaWYgeDIgPD0geDEgb3IgeTIgPD0geTE6IHJhaXNlIFZhbHVlRXJyb3IoImRldGVjdG9yIHJldHVybmVkIGVtcHR5IGJib3giKQogICAgcmV0dXJuIFt4MSwgeTEsIHgyLCB5Ml0KCmRlZiBfcGFkZGVkKGJveDogbGlzdFtpbnRdLCB3aWR0aDogaW50LCBoZWlnaHQ6IGludCwgcGFkZGluZzogZmxvYXQpIC0+IGxpc3RbaW50XToKICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKHBhZGRpbmcpIG9yIHBhZGRpbmcgPCAwOiByYWlzZSBWYWx1ZUVycm9yKCJwYWRkaW5nIG11c3QgYmUgZmluaXRlIGFuZCBub24tbmVnYXRpdmUiKQogICAgeDEseTEseDIseTIgPSBib3g7IGJ3LGJoPXgyLXgxLHkyLXkxOyBjeCxjeT0oeDEreDIpLzIsKHkxK3kyKS8yCiAgICByZXR1cm4gW21heCgwLGludChucC5mbG9vcihjeC1idyooLjUrcGFkZGluZykpKSksIG1heCgwLGludChucC5mbG9vcihjeS1iaCooLjUrcGFkZGluZykpKSksCiAgICAgICAgICAgIG1pbih3aWR0aCxpbnQobnAuY2VpbChjeCtidyooLjUrcGFkZGluZykpKSksIG1pbihoZWlnaHQsaW50KG5wLmNlaWwoY3krYmgqKC41K3BhZGRpbmcpKSkpXQoKZGVmIF9saW1pdChpbWFnZTogSW1hZ2UuSW1hZ2UsIG1heF9zaWRlOiBpbnQpIC0+IEltYWdlLkltYWdlOgogICAgaWYgbWF4X3NpZGUgPCAxOiByYWlzZSBWYWx1ZUVycm9yKCJtYXhfc2lkZSBtdXN0IGJlIHBvc2l0aXZlIikKICAgIHcsaD1pbWFnZS5zaXplOyBzY2FsZT1taW4oMS4sbWF4X3NpZGUvbWF4KHcsaCkpCiAgICByZXR1cm4gaW1hZ2UgaWYgc2NhbGUgPT0gMSBlbHNlIGltYWdlLnJlc2l6ZSgobWF4KDEsaW50KHcqc2NhbGUpKSxtYXgoMSxpbnQoaCpzY2FsZSkpKSxJbWFnZS5SZXNhbXBsaW5nLkxBTkNaT1MpCgpjbGFzcyBJbWFnZVBpcGVsaW5lOiAgIyBzdGFuZGFsb25lIE02IHByb2Nlc3Npbmcgb3JjaGVzdHJhdGlvbgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGdlbmVyYWxfcmVzdG9yZXI6IFJlc3RvcmVyLCBkZXRlY3RvcjogRGV0ZWN0b3IsIGZhY2VfcmVzdG9yZXI6IFJlc3RvcmVyLCBjb25maWc6IFBpcGVsaW5lQ29uZmlnfE5vbmU9Tm9uZSk6CiAgICAgICAgc2VsZi5nZW5lcmFsX3Jlc3RvcmVyLHNlbGYuZGV0ZWN0b3Isc2VsZi5mYWNlX3Jlc3RvcmVyPWdlbmVyYWxfcmVzdG9yZXIsZGV0ZWN0b3IsZmFjZV9yZXN0b3JlcgogICAgICAgIHNlbGYuY29uZmlnPWNvbmZpZyBvciBQaXBlbGluZUNvbmZpZygpCiAgICAgICAgaWYgc2VsZi5jb25maWcubm9fZmFjZV9wb2xpY3kgbm90IGluICgia2VlcCIsImVycm9yIik6IHJhaXNlIFZhbHVlRXJyb3IoIm5vX2ZhY2VfcG9saWN5IG11c3QgYmUga2VlcCBvciBlcnJvciIpCiAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoc2VsZi5jb25maWcucGFkZGluZykgb3Igc2VsZi5jb25maWcucGFkZGluZyA8IDA6IHJhaXNlIFZhbHVlRXJyb3IoInBhZGRpbmcgbXVzdCBiZSBmaW5pdGUgYW5kIG5vbi1uZWdhdGl2ZSIpCiAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoc2VsZi5jb25maWcuY29uZmlkZW5jZSkgb3Igbm90IDAgPD0gc2VsZi5jb25maWcuY29uZmlkZW5jZSA8PSAxOiByYWlzZSBWYWx1ZUVycm9yKCJjb25maWRlbmNlIG11c3QgYmUgaW4gWzAsIDFdIikKICAgICAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZShzZWxmLmNvbmZpZy5maWRlbGl0eSkgb3Igbm90IDAgPD0gc2VsZi5jb25maWcuZmlkZWxpdHkgPD0gMTogcmFpc2UgVmFsdWVFcnJvcigiZmlkZWxpdHkgbXVzdCBiZSBpbiBbMCwgMV0iKQogICAgICAgIGlmIHNlbGYuY29uZmlnLm1heF9zaWRlIDwgMTogcmFpc2UgVmFsdWVFcnJvcigibWF4X3NpZGUgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICAgICAgaWYgc2VsZi5jb25maWcubnVtcHlfY29sb3Igbm90IGluICgiUkdCIiwgIkJHUiIpOiByYWlzZSBWYWx1ZUVycm9yKCJudW1weV9jb2xvciBtdXN0IGJlIFJHQiBvciBCR1IiKQogICAgZGVmIHByb2Nlc3Moc2VsZiwgc291cmNlOiBzdHJ8UGF0aHxJbWFnZS5JbWFnZXxucC5uZGFycmF5KSAtPiBQaXBlbGluZVJlc3VsdDoKICAgICAgICByPVBpcGVsaW5lUmVzdWx0KGVuaGFuY2VkPU5vbmUsbW9kZWxfaWRzPXsiZ2VuZXJhbCI6c2VsZi5nZW5lcmFsX3Jlc3RvcmVyLm1vZGVsX2lkLCJkZXRlY3RvciI6c2VsZi5kZXRlY3Rvci5tb2RlbF9pZCwiZmFjZSI6c2VsZi5mYWNlX3Jlc3RvcmVyLm1vZGVsX2lkfSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIG9yaWdpbmFsPV9pbnB1dF9yZ2Ioc291cmNlLHNlbGYuY29uZmlnLm51bXB5X2NvbG9yKTsgci5pbnB1dF9kaW1lbnNpb25zPVtvcmlnaW5hbC53aWR0aCxvcmlnaW5hbC5oZWlnaHRdCiAgICAgICAgICAgIHQ9cGVyZl9jb3VudGVyKCk7IGVuaGFuY2VkPXNlbGYuZ2VuZXJhbF9yZXN0b3Jlci5yZXN0b3JlKG9yaWdpbmFsLmNvcHkoKSk7IHIuc3RhZ2VfdGltZXNfbXNbImdlbmVyYWxfcmVzdG9yYXRpb24iXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShlbmhhbmNlZCxJbWFnZS5JbWFnZSk6IHJhaXNlIFR5cGVFcnJvcigiZ2VuZXJhbCByZXN0b3JlciBtdXN0IHJldHVybiBQSUwuSW1hZ2UiKQogICAgICAgICAgICBlbmhhbmNlZD1lbmhhbmNlZC5jb252ZXJ0KCJSR0IiKQogICAgICAgICAgICBpZiBlbmhhbmNlZC5zaXplICE9IG9yaWdpbmFsLnNpemU6IHJhaXNlIFZhbHVlRXJyb3IoImdlbmVyYWwgcmVzdG9yYXRpb24gbXVzdCBwcmVzZXJ2ZSBkaW1lbnNpb25zOyBiYm94IG1hcHBpbmcgaXMgdW5kZWZpbmVkIikKICAgICAgICAgICAgci5lbmhhbmNlZD1lbmhhbmNlZDsgci5vdXRwdXRfZGltZW5zaW9ucz1bZW5oYW5jZWQud2lkdGgsZW5oYW5jZWQuaGVpZ2h0XQogICAgICAgICAgICB0PXBlcmZfY291bnRlcigpOyBkZXRlY3Rpb25zPXNlbGYuZGV0ZWN0b3IuZGV0ZWN0KGVuaGFuY2VkLmNvcHkoKSxzZWxmLmNvbmZpZy5jb25maWRlbmNlKTsgci5zdGFnZV90aW1lc19tc1siZmFjZV9kZXRlY3Rpb24iXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICB0PXBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGZvciBkIGluIGRldGVjdGlvbnM6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgY29uZmlkZW5jZT1mbG9hdChkWyJjb25maWRlbmNlIl0pCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoY29uZmlkZW5jZSkgb3Igbm90IDAgPD0gY29uZmlkZW5jZSA8PSAxOiByYWlzZSBWYWx1ZUVycm9yKCJkZXRlY3RvciBjb25maWRlbmNlIG11c3QgYmUgZmluaXRlIGFuZCBpbiBbMCwgMV0iKQogICAgICAgICAgICAgICAgICAgIGlmIGNvbmZpZGVuY2UgPCBzZWxmLmNvbmZpZy5jb25maWRlbmNlOiBjb250aW51ZQogICAgICAgICAgICAgICAgICAgIGJveD1feHl4eShkLmdldCgiYmJveF94eXh5IiksZW5oYW5jZWQud2lkdGgsZW5oYW5jZWQuaGVpZ2h0KTsgY2I9X3BhZGRlZChib3gsZW5oYW5jZWQud2lkdGgsZW5oYW5jZWQuaGVpZ2h0LHNlbGYuY29uZmlnLnBhZGRpbmcpCiAgICAgICAgICAgICAgICAgICAgci5mYWNlcy5hcHBlbmQoRmFjZVJlc3VsdChib3gsY2IsY29uZmlkZW5jZSxlbmhhbmNlZC5jcm9wKHR1cGxlKGNiKSkuY29udmVydCgiUkdCIikpKQogICAgICAgICAgICAgICAgZXhjZXB0IChLZXlFcnJvciwgVHlwZUVycm9yLCBWYWx1ZUVycm9yKSBhcyBleGM6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImRldGVjdG9yIHJldHVybmVkIG1hbGZvcm1lZCBmYWNlOiB7ZXhjfSIpIGZyb20gZXhjCiAgICAgICAgICAgIHIuc3RhZ2VfdGltZXNfbXNbImZhY2VfY3JvcHBpbmciXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICB0PXBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGZvciBmIGluIHIuZmFjZXM6CiAgICAgICAgICAgICAgICByZXN0b3JlZD1zZWxmLmZhY2VfcmVzdG9yZXIucmVzdG9yZShmLmNyb3Asc2VsZi5jb25maWcuZmlkZWxpdHkpCiAgICAgICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShyZXN0b3JlZCxJbWFnZS5JbWFnZSk6IHJhaXNlIFR5cGVFcnJvcigiZmFjZSByZXN0b3JlciBtdXN0IHJldHVybiBQSUwuSW1hZ2UiKQogICAgICAgICAgICAgICAgZi5jcm9wPV9saW1pdChyZXN0b3JlZC5jb252ZXJ0KCJSR0IiKSxzZWxmLmNvbmZpZy5tYXhfc2lkZSkKICAgICAgICAgICAgci5zdGFnZV90aW1lc19tc1siZmFjZV9yZXN0b3JhdGlvbiJdPShwZXJmX2NvdW50ZXIoKS10KSoxMDAwCiAgICAgICAgICAgIHQ9cGVyZl9jb3VudGVyKCk7IHIuZW5oYW5jZWQ9X2xpbWl0KGVuaGFuY2VkLHNlbGYuY29uZmlnLm1heF9zaWRlKTsgci5vdXRwdXRfZGltZW5zaW9ucz1bci5lbmhhbmNlZC53aWR0aCxyLmVuaGFuY2VkLmhlaWdodF0KICAgICAgICAgICAgci5zdGFnZV90aW1lc19tc1sib3V0cHV0X3NpemVfbGltaXQiXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICBpZiBub3Qgci5mYWNlcyBhbmQgc2VsZi5jb25maWcubm9fZmFjZV9wb2xpY3kgPT0gImVycm9yIjogcmFpc2UgUnVudGltZUVycm9yKCJub19mYWNlIikKICAgICAgICAgICAgci5zdGF0dXM9Im9rIiBpZiByLmZhY2VzIGVsc2UgIm5vX2ZhY2UiCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiByLnN0YXR1cyxyLmVycm9yPSJlcnJvciIsZiJ7dHlwZShlKS5fX25hbWVfX306IHtlfSIKICAgICAgICByZXR1cm4gcgoKZGVmIHByb2Nlc3NfYmF0Y2gocGlwZWxpbmU6IEltYWdlUGlwZWxpbmUsIHNvdXJjZXMpOgogICAgIiIiUHJvY2VzcyBldmVyeSBpbnB1dCBpbmRlcGVuZGVudGx5OyBhIGJhZCBmaWxlIGRvZXMgbm90IHN0b3AgdGhlIGJhdGNoLiIiIgogICAgcmV0dXJuIFsoc291cmNlLCBwaXBlbGluZS5wcm9jZXNzKHNvdXJjZSkpIGZvciBzb3VyY2UgaW4gc291cmNlc10K'))
source_setup = BUNDLE_ROOT / 'environment' / 'setup_emoca_source.sh'
run_quiet(['bash', str(source_setup), str(BUNDLE_ROOT), str(WORKDIR)],
          label='EMOCA source setup')


In [ ]:
# Stage the new frontalization modules and fetch the exact ADR-026 3DDFA-V2 source/artifacts.
import base64, hashlib, shutil
from pathlib import Path

PROJECT = WORKDIR / 'project'
FRONTALIZATION_PACKAGE = PROJECT / 'services' / 'frontalization'
FRONTALIZATION_PACKAGE.mkdir(parents=True, exist_ok=True)
(FRONTALIZATION_PACKAGE / '__init__.py').write_bytes(base64.b64decode('IiIiUHVibGljIGZyb250YWxpemF0aW9uIHR5cGVzIGFuZCBhY2Nlc3Nvci4iIiIKZnJvbSAudGRmYV92Ml93cmFwcGVyIGltcG9ydCBGcm9udGFsaXphdGlvbkVycm9yLCBGcm9udGFsaXphdGlvblJlc3VsdCwgVERERkFWMkZyb250YWxpemVyLCBnZXRfdGRkZmFfdjJfZnJvbnRhbGl6ZXIKCl9fYWxsX18gPSBbIkZyb250YWxpemF0aW9uRXJyb3IiLCAiRnJvbnRhbGl6YXRpb25SZXN1bHQiLCAiVERERkFWMkZyb250YWxpemVyIiwgImdldF90ZGRmYV92Ml9mcm9udGFsaXplciJdCg=='))
(FRONTALIZATION_PACKAGE / 'tdfa_v2_wrapper.py').write_bytes(base64.b64decode('IiIiQ1BVIDNEREZBLVYyIGZhY2UgZnJvbnRhbGl6YXRpb24gZm9yIHRoZSBvcHQtaW4gd2ViIGV4cGVyaW1lbnQuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0aHJlYWRpbmcgaW1wb3J0IExvY2sKZnJvbSB0aW1lIGltcG9ydCBwZXJmX2NvdW50ZXIKaW1wb3J0IGltcG9ydGxpYi51dGlsCmltcG9ydCBwaWNrbGUKCmltcG9ydCBjdjIKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gUElMIGltcG9ydCBJbWFnZQoKUFJPSkVDVF9ST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KTU9ERUxfRElSID0gUFJPSkVDVF9ST09UIC8gIm1vZGVsIiAvICIzZGRmYV92MiIKVVBTVFJFQU1fUk9PVCA9IFBST0pFQ1RfUk9PVCAvICJ0aGlyZF9wYXJ0eSIgLyAiM0RERkFfVjIiCgoKY2xhc3MgRnJvbnRhbGl6YXRpb25FcnJvcihSdW50aW1lRXJyb3IpOgogICAgIiIiQSBtb2RlbCBsb2FkLCBmYWNlLWZpdCwgb3Igd2FycCBmYWlsdXJlLiIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvZGU6IHN0ciwgbWVzc2FnZTogc3RyKToKICAgICAgICBpZiBjb2RlIG5vdCBpbiB7Im5vX2ZhY2UiLCAiZml0X2ZhaWxlZCIsICJsb2FkX2ZhaWxlZCJ9OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbnZhbGlkIGZyb250YWxpemF0aW9uIGVycm9yIGNvZGUiKQogICAgICAgIHN1cGVyKCkuX19pbml0X18obWVzc2FnZSkKICAgICAgICBzZWxmLmNvZGUgPSBjb2RlCiAgICAgICAgc2VsZi5tZXNzYWdlID0gbWVzc2FnZQoKCkBkYXRhY2xhc3MKY2xhc3MgRnJvbnRhbGl6YXRpb25SZXN1bHQ6CiAgICBpbWFnZTogSW1hZ2UuSW1hZ2UKICAgIHlhd19kZWc6IGZsb2F0CiAgICBwaXRjaF9kZWc6IGZsb2F0CiAgICByb2xsX2RlZzogZmxvYXQKICAgIG9jY2x1ZGVkX2ZyYWN0aW9uOiBmbG9hdAogICAgdGltZV9tczogZmxvYXQKICAgIG1ldGhvZDogc3RyID0gIjNkZGZhX3YyIgoKCmRlZiBfcGlja2xlKHBhdGg6IFBhdGgpOgogICAgdHJ5OgogICAgICAgIHdpdGggcGF0aC5vcGVuKCJyYiIpIGFzIHN0cmVhbToKICAgICAgICAgICAgcmV0dXJuIHBpY2tsZS5sb2FkKHN0cmVhbSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgd2l0aCBwYXRoLm9wZW4oInJiIikgYXMgc3RyZWFtOgogICAgICAgICAgICByZXR1cm4gcGlja2xlLmxvYWQoc3RyZWFtLCBlbmNvZGluZz0ibGF0aW4xIikKCgpkZWYgX3Bvc2VfYW5nbGVzKHJvdGF0aW9uOiBucC5uZGFycmF5KSAtPiB0dXBsZVtmbG9hdCwgZmxvYXQsIGZsb2F0XToKICAgICMgUG9ydCBvZiB0aGlyZF9wYXJ0eS8zRERGQV9WMi91dGlscy9wb3NlLnB5OjE4LTYxIChQMnNSdCBhbmQgbWF0cml4MmFuZ2xlKS4KICAgIHIxLCByMiA9IHJvdGF0aW9uWzA6MV0sIHJvdGF0aW9uWzE6Ml0KICAgIHIxID0gcjEgLyBtYXgoZmxvYXQobnAubGluYWxnLm5vcm0ocjEpKSwgMWUtMTIpCiAgICByMiA9IHIyIC8gbWF4KGZsb2F0KG5wLmxpbmFsZy5ub3JtKHIyKSksIDFlLTEyKQogICAgciA9IG5wLmNvbmNhdGVuYXRlKChyMSwgcjIsIG5wLmNyb3NzKHIxLCByMikpLCBheGlzPTApCiAgICBpZiByWzIsIDBdID4gLjk5ODoKICAgICAgICB6LCB4LCB5ID0gMC4wLCBucC5waSAvIDIsIG5wLmFyY3RhbjIoLXJbMCwgMV0sIC1yWzAsIDJdKQogICAgZWxpZiByWzIsIDBdIDwgLS45OTg6CiAgICAgICAgeiwgeCwgeSA9IDAuMCwgLW5wLnBpIC8gMiwgbnAuYXJjdGFuMihyWzAsIDFdLCByWzAsIDJdKQogICAgZWxzZToKICAgICAgICB4ID0gbnAuYXJjc2luKHJbMiwgMF0pCiAgICAgICAgeSA9IG5wLmFyY3RhbjIoclsyLCAxXSAvIG5wLmNvcyh4KSwgclsyLCAyXSAvIG5wLmNvcyh4KSkKICAgICAgICB6ID0gbnAuYXJjdGFuMihyWzEsIDBdIC8gbnAuY29zKHgpLCByWzAsIDBdIC8gbnAuY29zKHgpKQogICAgcmV0dXJuIHR1cGxlKGZsb2F0KHYgKiAxODAgLyBucC5waSkgZm9yIHYgaW4gKHgsIHksIHopKQoKCmNsYXNzIFREREZBVjJGcm9udGFsaXplcjoKICAgICIiIkxhenkgQ1BVLW9ubHkgTW9iaWxlTmV0IDNETU0gZml0dGVyIGFuZCBpbWFnZS1zcGFjZSBmcm9udGFsaXplci4iIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtb2RlbF9kaXI6IHN0ciB8IFBhdGggPSBNT0RFTF9ESVIsCiAgICAgICAgICAgICAgICAgdXBzdHJlYW1fcm9vdDogc3RyIHwgUGF0aCA9IFVQU1RSRUFNX1JPT1QsIGRldmljZTogc3RyID0gImNwdSIpOgogICAgICAgIGlmIGRldmljZSAhPSAiY3B1IjoKICAgICAgICAgICAgcmFpc2UgRnJvbnRhbGl6YXRpb25FcnJvcigibG9hZF9mYWlsZWQiLCAiM0RERkEtVjIgZnJvbnRhbGl6ZXIgb25seSBzdXBwb3J0cyBDUFUiKQogICAgICAgIHNlbGYubW9kZWxfZGlyID0gUGF0aChtb2RlbF9kaXIpCiAgICAgICAgc2VsZi51cHN0cmVhbV9yb290ID0gUGF0aCh1cHN0cmVhbV9yb290KQogICAgICAgIHNlbGYuX2xvYWRlZCA9IEZhbHNlCiAgICAgICAgc2VsZi5fbG9jayA9IExvY2soKQoKICAgIGRlZiBfbG9hZChzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYuX2xvYWRlZDoKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBpZiBzZWxmLl9sb2FkZWQ6CiAgICAgICAgICAgICAgICByZXR1cm4KICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgICAgICAgICBtb2R1bGVfcGF0aCA9IHNlbGYudXBzdHJlYW1fcm9vdCAvICJtb2RlbHMiIC8gIm1vYmlsZW5ldF92MS5weSIKICAgICAgICAgICAgICAgIHNwZWMgPSBpbXBvcnRsaWIudXRpbC5zcGVjX2Zyb21fZmlsZV9sb2NhdGlvbigibWEzZF90ZGRmYV9tb2JpbGVuZXRfdjEiLCBtb2R1bGVfcGF0aCkKICAgICAgICAgICAgICAgIGlmIHNwZWMgaXMgTm9uZSBvciBzcGVjLmxvYWRlciBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmImNhbm5vdCBsb2FkIHttb2R1bGVfcGF0aH0iKQogICAgICAgICAgICAgICAgbW9kdWxlID0gaW1wb3J0bGliLnV0aWwubW9kdWxlX2Zyb21fc3BlYyhzcGVjKQogICAgICAgICAgICAgICAgc3BlYy5sb2FkZXIuZXhlY19tb2R1bGUobW9kdWxlKQogICAgICAgICAgICAgICAgc2VsZi5tb2RlbCA9IG1vZHVsZS5tb2JpbGVuZXQod2lkZW5fZmFjdG9yPTEuMCwgbnVtX2NsYXNzZXM9NjIpLmNwdSgpLmV2YWwoKQogICAgICAgICAgICAgICAgY2hlY2twb2ludCA9IHRvcmNoLmxvYWQoc2VsZi5tb2RlbF9kaXIgLyAibWIxXzEyMHgxMjAucHRoIiwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICAgICAgICAgICAgICBzdGF0ZSA9IGNoZWNrcG9pbnQuZ2V0KCJzdGF0ZV9kaWN0IiwgY2hlY2twb2ludCkgaWYgaXNpbnN0YW5jZShjaGVja3BvaW50LCBkaWN0KSBlbHNlIGNoZWNrcG9pbnQKICAgICAgICAgICAgICAgICMgVXBzdHJlYW0gdXRpbHMvdGRkZmFfdXRpbC5weTozMy0zOCBkcm9wcyBmY19sbSBhbmQgbWFwcyBmY19wYXJhbSB0byBmYy4KICAgICAgICAgICAgICAgIHN0YXRlID0ge2sucmVtb3ZlcHJlZml4KCJtb2R1bGUuIik6IHYgZm9yIGssIHYgaW4gc3RhdGUuaXRlbXMoKQogICAgICAgICAgICAgICAgICAgICAgICAgaWYgbm90IGsucmVtb3ZlcHJlZml4KCJtb2R1bGUuIikuc3RhcnRzd2l0aCgiZmNfbG0uIil9CiAgICAgICAgICAgICAgICBzdGF0ZSA9IHtrLnJlcGxhY2UoImZjX3BhcmFtLiIsICJmYy4iLCAxKTogdiBmb3IgaywgdiBpbiBzdGF0ZS5pdGVtcygpfQogICAgICAgICAgICAgICAgc2VsZi5tb2RlbC5sb2FkX3N0YXRlX2RpY3Qoc3RhdGUsIHN0cmljdD1UcnVlKQogICAgICAgICAgICAgICAgYmZtID0gX3BpY2tsZShzZWxmLm1vZGVsX2RpciAvICJiZm1fbm9uZWNrX3YzLnBrbCIpCiAgICAgICAgICAgICAgICBzZWxmLnUgPSBucC5hc2FycmF5KGJmbVsidSJdLCBkdHlwZT1ucC5mbG9hdDMyKS5yZXNoYXBlKC0xKQogICAgICAgICAgICAgICAgc2VsZi53X3NocCA9IG5wLmFzYXJyYXkoYmZtWyJ3X3NocCJdLCBkdHlwZT1ucC5mbG9hdDMyKVs6LCA6NDBdCiAgICAgICAgICAgICAgICBzZWxmLndfZXhwID0gbnAuYXNhcnJheShiZm1bIndfZXhwIl0sIGR0eXBlPW5wLmZsb2F0MzIpWzosIDoxMF0KICAgICAgICAgICAgICAgIHNlbGYudHJpID0gbnAuYXNhcnJheShfcGlja2xlKHNlbGYubW9kZWxfZGlyIC8gInRyaS5wa2wiKSwgZHR5cGU9bnAuaW50MzIpCiAgICAgICAgICAgICAgICBpZiBzZWxmLnRyaS5uZGltID09IDIgYW5kIHNlbGYudHJpLnNoYXBlWzBdID09IDM6CiAgICAgICAgICAgICAgICAgICAgc2VsZi50cmkgPSBzZWxmLnRyaS5UCiAgICAgICAgICAgICAgICAjIHVwc3RyZWFtIEJGTSB0cmlhbmdsZXMgYXJlIG9uZS1iYXNlZDsgbm9ybWFsaXplIG9uY2UgZm9yIE51bVB5LgogICAgICAgICAgICAgICAgaWYgc2VsZi50cmkubWluKCkgPT0gMToKICAgICAgICAgICAgICAgICAgICBzZWxmLnRyaSA9IHNlbGYudHJpIC0gMQogICAgICAgICAgICAgICAgc3RhdHMgPSBfcGlja2xlKHNlbGYubW9kZWxfZGlyIC8gInBhcmFtX21lYW5fc3RkXzYyZF8xMjB4MTIwLnBrbCIpCiAgICAgICAgICAgICAgICBzZWxmLnBhcmFtX21lYW4gPSBucC5hc2FycmF5KHN0YXRzWyJtZWFuIl0sIGR0eXBlPW5wLmZsb2F0MzIpLnJlc2hhcGUoLTEpCiAgICAgICAgICAgICAgICBzZWxmLnBhcmFtX3N0ZCA9IG5wLmFzYXJyYXkoc3RhdHNbInN0ZCJdLCBkdHlwZT1ucC5mbG9hdDMyKS5yZXNoYXBlKC0xKQogICAgICAgICAgICAgICAgc2VsZi50b3JjaCA9IHRvcmNoCiAgICAgICAgICAgICAgICBzZWxmLl9sb2FkZWQgPSBUcnVlCiAgICAgICAgICAgIGV4Y2VwdCBGcm9udGFsaXphdGlvbkVycm9yOgogICAgICAgICAgICAgICAgcmFpc2UKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgICAgICByYWlzZSBGcm9udGFsaXphdGlvbkVycm9yKCJsb2FkX2ZhaWxlZCIsIGYiQ291bGQgbm90IGxvYWQgM0RERkEtVjI6IHt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSIpIGZyb20gZXhjCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIF9yb2kod2lkdGg6IGludCwgaGVpZ2h0OiBpbnQpIC0+IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXQsIGZsb2F0XToKICAgICAgICAjIFBvcnQgb2YgdGhpcmRfcGFydHkvM0RERkFfVjIvdXRpbHMvZnVuY3Rpb25zLnB5Ojg1LTEwMC4KICAgICAgICBvbGRfc2l6ZSA9ICh3aWR0aCArIGhlaWdodCkgLyAyCiAgICAgICAgY3ggPSB3aWR0aCAvIDIKICAgICAgICBjeSA9IGhlaWdodCAvIDIgKyBvbGRfc2l6ZSAqIC4xNAogICAgICAgIHNpemUgPSBpbnQob2xkX3NpemUgKiAxLjU4KQogICAgICAgIHJldHVybiBjeC1zaXplLzIsIGN5LXNpemUvMiwgY3grc2l6ZS8yLCBjeStzaXplLzIKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX2Nyb3AoaW1hZ2U6IG5wLm5kYXJyYXksIHJvaTogdHVwbGVbZmxvYXQsIGZsb2F0LCBmbG9hdCwgZmxvYXRdKSAtPiBucC5uZGFycmF5OgogICAgICAgICMgUG9ydCBvZiB0aGlyZF9wYXJ0eS8zRERGQV9WMi91dGlscy9mdW5jdGlvbnMucHk6MjMtNDcsIGluY2x1ZGluZyB6ZXJvIHBhZGRpbmcuCiAgICAgICAgeDEsIHkxLCB4MiwgeTIgPSAoaW50KHJvdW5kKHgpKSBmb3IgeCBpbiByb2kpCiAgICAgICAgaCwgdyA9IGltYWdlLnNoYXBlWzoyXQogICAgICAgIG91dCA9IG5wLnplcm9zKCh5Mi15MSwgeDIteDEsIDMpLCBucC51aW50OCkKICAgICAgICBzeCwgc3ksIGV4LCBleSA9IG1heCgwLHgxKSwgbWF4KDAseTEpLCBtaW4odyx4MiksIG1pbihoLHkyKQogICAgICAgIGlmIGV4ID4gc3ggYW5kIGV5ID4gc3k6CiAgICAgICAgICAgIG91dFtzeS15MTpleS15MSwgc3gteDE6ZXgteDFdID0gaW1hZ2Vbc3k6ZXksIHN4OmV4XQogICAgICAgIHJldHVybiBvdXQKCiAgICBkZWYgZXN0aW1hdGUoc2VsZiwgaW1hZ2U6IEltYWdlLkltYWdlKSAtPiB0dXBsZVtucC5uZGFycmF5LCB0dXBsZVtmbG9hdCwgZmxvYXQsIGZsb2F0XV06CiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UoaW1hZ2UsIEltYWdlLkltYWdlKSBvciBpbWFnZS53aWR0aCA8IDIgb3IgaW1hZ2UuaGVpZ2h0IDwgMjoKICAgICAgICAgICAgcmFpc2UgRnJvbnRhbGl6YXRpb25FcnJvcigibm9fZmFjZSIsICJJbnB1dCBjcm9wIGlzIGVtcHR5IG9yIHRvbyBzbWFsbCIpCiAgICAgICAgcmdiID0gbnAuYXNhcnJheShpbWFnZS5jb252ZXJ0KCJSR0IiKSwgZHR5cGU9bnAudWludDgpCiAgICAgICAgaWYgZmxvYXQocmdiLnN0ZCgpKSA8IC41OgogICAgICAgICAgICByYWlzZSBGcm9udGFsaXphdGlvbkVycm9yKCJub19mYWNlIiwgIlVuaWZvcm0gY3JvcCBoYXMgbm8gdXNhYmxlIGZhY2UgZGV0YWlsIikKICAgICAgICBzZWxmLl9sb2FkKCkKICAgICAgICByb2kgPSBzZWxmLl9yb2koaW1hZ2Uud2lkdGgsIGltYWdlLmhlaWdodCkKICAgICAgICAjIFVwc3RyZWFtIHVzZXMgY3YyIGlucHV0IChCR1IpIGFuZCBsZWF2ZXMgdGhhdCBjaGFubmVsIG9yZGVyIGZvciB0aGUgbmV0LgogICAgICAgIGJnciA9IGN2Mi5jdnRDb2xvcihyZ2IsIGN2Mi5DT0xPUl9SR0IyQkdSKQogICAgICAgIGNyb3AgPSBjdjIucmVzaXplKHNlbGYuX2Nyb3AoYmdyLCByb2kpLCAoMTIwLDEyMCksIGludGVycG9sYXRpb249Y3YyLklOVEVSX0xJTkVBUikKICAgICAgICBpbnAgPSBzZWxmLnRvcmNoLmZyb21fbnVtcHkoY3JvcC50cmFuc3Bvc2UoMiwwLDEpLmNvcHkoKSkuZmxvYXQoKS5zdWJfKDEyNy41KS5kaXZfKDEyOCkudW5zcXVlZXplKDApCiAgICAgICAgd2l0aCBzZWxmLnRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgICAgIHByZWQgPSBzZWxmLm1vZGVsKGlucCkuc3F1ZWV6ZSgwKS5jcHUoKS5udW1weSgpLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgIHBhcmFtID0gcHJlZCAqIHNlbGYucGFyYW1fc3RkICsgc2VsZi5wYXJhbV9tZWFuCiAgICAgICAgaWYgcGFyYW0uc2hhcGUgIT0gKDYyLCkgb3Igbm90IG5wLmlzZmluaXRlKHBhcmFtKS5hbGwoKToKICAgICAgICAgICAgcmFpc2UgRnJvbnRhbGl6YXRpb25FcnJvcigibm9fZmFjZSIsICIzRERGQS1WMiByZXR1cm5lZCBpbnZhbGlkIHBhcmFtZXRlcnMiKQogICAgICAgIHJvdGF0aW9uID0gcGFyYW1bOjEyXS5yZXNoYXBlKDMsNClbOiw6M10KICAgICAgICByZXR1cm4gcGFyYW0sIF9wb3NlX2FuZ2xlcyhyb3RhdGlvbikKCiAgICBkZWYgX3JlY29uc3RydWN0KHNlbGYsIHBhcmFtOiBucC5uZGFycmF5LCByb2k6IHR1cGxlW2Zsb2F0LGZsb2F0LGZsb2F0LGZsb2F0XSwgc2l6ZTogdHVwbGVbaW50LGludF0pIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgIyBQb3J0IG9mIHRoaXJkX3BhcnR5LzNEREZBX1YyL1REREZBLnB5OjEyNC0xMzkgYW5kIHV0aWxzL3RkZGZhX3V0aWwucHk6NjQtNzcuCiAgICAgICAgY2FtZXJhID0gcGFyYW1bOjEyXS5yZXNoYXBlKDMsNCkKICAgICAgICByb3RhdGlvbiwgb2Zmc2V0ID0gY2FtZXJhWzosOjNdLCBjYW1lcmFbOiwzOjRdCiAgICAgICAgc2hhcGUgPSBwYXJhbVsxMjo1Ml0ucmVzaGFwZSg0MCwxKQogICAgICAgIGV4cCA9IHBhcmFtWzUyOjYyXS5yZXNoYXBlKDEwLDEpCiAgICAgICAgbW9kZWwgPSBzZWxmLnVbOixOb25lXSArIHNlbGYud19zaHAgQCBzaGFwZSArIHNlbGYud19leHAgQCBleHAKICAgICAgICBwb2ludHMgPSAocm90YXRpb24gQCBtb2RlbC5yZXNoYXBlKDMsLTEsb3JkZXI9IkYiKSArIG9mZnNldCkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgbiA9IDEyMAogICAgICAgIHBvaW50c1swXSAtPSAxCiAgICAgICAgcG9pbnRzWzJdIC09IDEKICAgICAgICBwb2ludHNbMV0gPSBuIC0gcG9pbnRzWzFdCiAgICAgICAgc3gsc3ksZXgsZXkgPSByb2kKICAgICAgICBzY2FsZV94LHNjYWxlX3k9KGV4LXN4KS9uLChleS1zeSkvbgogICAgICAgIHBvaW50c1swXSA9IHBvaW50c1swXSpzY2FsZV94K3N4CiAgICAgICAgcG9pbnRzWzFdID0gcG9pbnRzWzFdKnNjYWxlX3krc3kKICAgICAgICBwb2ludHNbMl0gKj0gKHNjYWxlX3grc2NhbGVfeSkvMgogICAgICAgIHBvaW50c1syXSAtPSBwb2ludHNbMl0ubWluKCkKICAgICAgICByZXR1cm4gcG9pbnRzCgogICAgZGVmIGZyb250YWxpemUoc2VsZiwgaW1hZ2U6IEltYWdlLkltYWdlKSAtPiBGcm9udGFsaXphdGlvblJlc3VsdDoKICAgICAgICBzdGFydGVkID0gcGVyZl9jb3VudGVyKCkKICAgICAgICBwYXJhbSwgcG9zZSA9IHNlbGYuZXN0aW1hdGUoaW1hZ2UpCiAgICAgICAgd2lkdGgsaGVpZ2h0PWltYWdlLnNpemUKICAgICAgICByb2k9c2VsZi5fcm9pKHdpZHRoLGhlaWdodCkKICAgICAgICBwdHM9c2VsZi5fcmVjb25zdHJ1Y3QocGFyYW0scm9pLGltYWdlLnNpemUpCiAgICAgICAgeHl6PXB0cy5UCiAgICAgICAgeCx5LHo9eHl6WzosMF0seHl6WzosMV0seHl6WzosMl0KICAgICAgICBpZiBsZW4oeCk8MyBvciBub3QgbnAuaXNmaW5pdGUoeHl6KS5hbGwoKToKICAgICAgICAgICAgcmFpc2UgRnJvbnRhbGl6YXRpb25FcnJvcigibm9fZmFjZSIsICJQcm9qZWN0ZWQgM0QgZmFjZSBpcyBpbnZhbGlkIikKICAgICAgICBpbnNpZGU9KHg+PTApJih4PHdpZHRoKSYoeT49MCkmKHk8aGVpZ2h0KQogICAgICAgIGlmIGZsb2F0KGluc2lkZS5tZWFuKCkpIDwgLjIwOgogICAgICAgICAgICByYWlzZSBGcm9udGFsaXphdGlvbkVycm9yKCJub19mYWNlIiwgIlByb2plY3RlZCBmYWNlIGlzIG1vc3RseSBvdXRzaWRlIHRoZSBjcm9wIikKICAgICAgICB0cmk9c2VsZi50cmkKICAgICAgICBpZiB0cmkuc2l6ZSA9PSAwIG9yIHRyaS5tYXgoKSA+PSBsZW4oeCk6CiAgICAgICAgICAgIHJhaXNlIEZyb250YWxpemF0aW9uRXJyb3IoImZpdF9mYWlsZWQiLCAiQkZNIHRvcG9sb2d5IGlzIGludmFsaWQiKQogICAgICAgIGZhY2VzPXh5elt0cmldCiAgICAgICAgbm9ybWFscz1ucC5jcm9zcyhmYWNlc1s6LDFdLWZhY2VzWzosMF0sZmFjZXNbOiwyXS1mYWNlc1s6LDBdKQogICAgICAgICMgQWZ0ZXIgc2ltaWxhcl90cmFuc2Zvcm0gKHkgZmxpcHBlZCwgeiBzaGlmdGVkIHRvID49IDApIGNhbWVyYS1mYWNpbmcgbm9ybWFscyBoYXZlCiAgICAgICAgIyAreiBhbmQgdGhlIG5vc2UgdGlwIGhhcyB0aGUgbGFyZ2VzdCB6IChtZWFzdXJlZCBvbiByZWFsIGNyb3BzOiA4NS05MiAlIG9mCiAgICAgICAgIyB0cmlhbmdsZXMgZmFjZSAreiBvbiBmcm9udGFsIGZhY2VzKS4KICAgICAgICBmYWNpbmc9bm9ybWFsc1s6LDJdPjAKICAgICAgICBkZWdyZWU9bnAuYmluY291bnQodHJpLnJlc2hhcGUoLTEpLG1pbmxlbmd0aD1sZW4oeCkpLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgIGRlZiB2ZXJ0ZXhfbWVhbihwZXJfdHJpYW5nbGU6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgICAgIHRvdGFsPW5wLnplcm9zKGxlbih4KSxucC5mbG9hdDMyKQogICAgICAgICAgICBucC5hZGQuYXQodG90YWwsdHJpLnJlc2hhcGUoLTEpLG5wLnJlcGVhdChwZXJfdHJpYW5nbGUuYXN0eXBlKG5wLmZsb2F0MzIpLDMpKQogICAgICAgICAgICByZXR1cm4gdG90YWwvbnAubWF4aW11bShkZWdyZWUsMSkKICAgICAgICAjIEEgdmVydGV4IGlzIHZpc2libGUgd2hlbiBtb3N0IG9mIGl0cyB0cmlhbmdsZXMgZmFjZSB0aGUgY2FtZXJhIChub3QganVzdCBvbmUpLgogICAgICAgIHZpc2libGU9KHZlcnRleF9tZWFuKGZhY2luZyk+PS41KSZpbnNpZGUKICAgICAgICAjIENvYXJzZSB6LWJ1ZmZlciAobGFyZ2VyIHogPSBuZWFyZXIpIG9ubHkgZm9yIGNsZWFyIHNlbGYtb2NjbHVzaW9uIChlLmcuIHRoZSBub3NlIGF0CiAgICAgICAgIyBsYXJnZSB5YXcpLiBDZWxsIH4zICUgb2YgZmFjZSB3aWR0aCwgdG9sZXJhbmNlIDIwICUgb2YgZmFjZSBkZXB0aDogdGlnaHRlciB0ZXN0cwogICAgICAgICMgZmxhZyBzbG9wZWQgc2tpbiBhcyBoaWRkZW4gYW5kIHNwZWNrbGUgdGhlIHdhcnAgKHNlZW4gaW4gUUEsIDIwMjYtMTAtMDMpLgogICAgICAgIGNlbGw9bWF4KDIsaW50KHJvdW5kKGZsb2F0KG5wLnB0cCh4W2luc2lkZV0pKSouMDMpKSkKICAgICAgICBneD1ucC5jbGlwKCh4Ly9jZWxsKS5hc3R5cGUoaW50KSwwLCh3aWR0aC0xKS8vY2VsbCk7IGd5PW5wLmNsaXAoKHkvL2NlbGwpLmFzdHlwZShpbnQpLDAsKGhlaWdodC0xKS8vY2VsbCkKICAgICAgICB6YnVmPW5wLmZ1bGwoKChoZWlnaHQtMSkvL2NlbGwrMSwod2lkdGgtMSkvL2NlbGwrMSksLW5wLmluZixucC5mbG9hdDMyKQogICAgICAgIG5wLm1heGltdW0uYXQoemJ1ZiwoZ3lbaW5zaWRlXSxneFtpbnNpZGVdKSx6W2luc2lkZV0pCiAgICAgICAgdmlzaWJsZSAmPSB6ID49IHpidWZbZ3ksZ3hdIC0gbWF4KDIuLGZsb2F0KG5wLnB0cCh6KSkqLjIwKQogICAgICAgICMgT25lIHNtb290aGluZyBwYXNzIG92ZXIgdGhlIG1lc2ggc28gaGlkZGVuIHJlZ2lvbnMgYXJlIGNvbnRpZ3VvdXMgcGF0Y2hlcy4KICAgICAgICB2aXNpYmxlPSh2ZXJ0ZXhfbWVhbih2aXNpYmxlW3RyaV0ubWVhbihheGlzPTEpKT49LjUpJmluc2lkZQogICAgICAgIG9jY2x1ZGVkPTEtZmxvYXQodmlzaWJsZS5zdW0oKS9tYXgoMSxsZW4oeCkpKQogICAgICAgICMgUmVwcm9qZWN0IHRoZSBzYW1lIHNoYXBlL2V4cHJlc3Npb24gd2l0aCBpZGVudGl0eSByb3RhdGlvbjsgcG9ydCBvZiBURERGQS5weTpyZWNvbl92ZXJzLgogICAgICAgIGNhbWVyYSA9IHBhcmFtWzoxMl0ucmVzaGFwZSgzLDQpCiAgICAgICAgcm90YXRpb24sIG9mZnNldCA9IGNhbWVyYVs6LDozXSwgY2FtZXJhWzosMzo0XQogICAgICAgIHNoYXBlID0gcGFyYW1bMTI6NTJdLnJlc2hhcGUoNDAsMSkKICAgICAgICBleHAgPSBwYXJhbVs1Mjo2Ml0ucmVzaGFwZSgxMCwxKQogICAgICAgIG1vZGVsID0gc2VsZi51WzosTm9uZV0gKyBzZWxmLndfc2hwIEAgc2hhcGUgKyBzZWxmLndfZXhwIEAgZXhwCiAgICAgICAgbW9kZWwgPSBtb2RlbC5yZXNoYXBlKDMsLTEsb3JkZXI9IkYiKQogICAgICAgIGNhbWVyYV9zY2FsZSA9IChucC5saW5hbGcubm9ybShyb3RhdGlvblswLDozXSkgKyBucC5saW5hbGcubm9ybShyb3RhdGlvblsxLDozXSkpIC8gMgogICAgICAgIGZyb250YWxfcm9pID0gKGNhbWVyYV9zY2FsZSptb2RlbFswXStvZmZzZXRbMF0sIGNhbWVyYV9zY2FsZSptb2RlbFsxXStvZmZzZXRbMV0pCiAgICAgICAgZngwLGZ5MD1mcm9udGFsX3JvaQogICAgICAgIHN4LHN5LGV4LGV5PXJvaQogICAgICAgIHNjYWxlX3gsc2NhbGVfeT0oZXgtc3gpLzEyMCwoZXktc3kpLzEyMAogICAgICAgIGZ4MD0oZngwLTEpKnNjYWxlX3grc3gKICAgICAgICBmeTA9KDEyMC1meTApKnNjYWxlX3krc3kgICMgc2FtZSB5IGZsaXAgYXMgc2ltaWxhcl90cmFuc2Zvcm0KICAgICAgICAjIEtlZXAgdGhlIGNhbWVyYSBzY2FsZSAoYSB5YXdlZCBmYWNlIGlzIG5hcnJvd2VyLCBub3Qgc21hbGxlcik7IG9ubHkgcmUtY2VudHJlIGl0CiAgICAgICAgIyBvbiB0aGUgdmlzaWJsZSBmYWNlIHNvIHRoZSBmcm9udGFsIGZhY2Ugc3RheXMgaW5zaWRlIHRoZSBjcm9wLgogICAgICAgIGZ4PWZ4MC1mbG9hdChmeDAubWVhbigpKStmbG9hdCh4W2luc2lkZV0ubWVhbigpKQogICAgICAgIGZ5PWZ5MC1mbG9hdChmeTAubWVhbigpKStmbG9hdCh5W2luc2lkZV0ubWVhbigpKQogICAgICAgICMgU2FtcGxlIHZpc2libGUgdmVydGljZXMgZnJvbSBpbnB1dC4gSGlkZGVuIHZlcnRpY2VzIHNvdXJjZSB0aGVpciBiaWxhdGVyYWwgQkZNIHBhcnRuZXIuCiAgICAgICAgc291cmNlX3g9eC5jb3B5KCk7IHNvdXJjZV95PXkuY29weSgpCiAgICAgICAgbWlycm9yPXNlbGYuX21pcnJvcl9wYWlycygpCiAgICAgICAgaGlkZGVuPX52aXNpYmxlCiAgICAgICAgcGFpcj1taXJyb3JbaGlkZGVuXQogICAgICAgIHZhbGlkPXBhaXI+PTAKICAgICAgICBpZHM9bnAuZmxhdG5vbnplcm8oaGlkZGVuKVt2YWxpZF07IHBhcnRuZXJzPXBhaXJbdmFsaWRdCiAgICAgICAgc291cmNlX3hbaWRzXT14W3BhcnRuZXJzXTsgc291cmNlX3lbaWRzXT15W3BhcnRuZXJzXQogICAgICAgIHBvaW50cz1ucC5jb2x1bW5fc3RhY2soKGZ4LGZ5KSkKICAgICAgICBzb3VyY2U9bnAuY29sdW1uX3N0YWNrKChzb3VyY2VfeCxzb3VyY2VfeSkpCiAgICAgICAgIyBEZWNpbWF0ZSBkZXRlcm1pbmlzdGljYWxseSBmb3IgYSBib3VuZGVkIERlbGF1bmF5IGJhY2t3YXJkIG1hcC4KICAgICAgICBzdHJpZGU9bWF4KDEsaW50KG5wLmNlaWwobGVuKHBvaW50cykvOTAwMCkpKQogICAgICAgIGNob3Nlbj1ucC5hcmFuZ2UoMCxsZW4ocG9pbnRzKSxzdHJpZGUpCiAgICAgICAgY2hvc2VuPWNob3NlbltucC5pc2Zpbml0ZShwb2ludHNbY2hvc2VuXSkuYWxsKGF4aXM9MSldCiAgICAgICAgZnJvbSBzY2lweS5pbnRlcnBvbGF0ZSBpbXBvcnQgTGluZWFyTkRJbnRlcnBvbGF0b3IKICAgICAgICB0cnk6CiAgICAgICAgICAgIHl5LHh4PW5wLm1ncmlkWzA6aGVpZ2h0LDA6d2lkdGhdCiAgICAgICAgICAgIGludGVycD1MaW5lYXJOREludGVycG9sYXRvcihwb2ludHNbY2hvc2VuXSxzb3VyY2VbY2hvc2VuXSxmaWxsX3ZhbHVlPS0xKQogICAgICAgICAgICBtYXBwaW5nPW5wLmFzYXJyYXkoaW50ZXJwKHh4LHl5KSxkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICByYWlzZSBGcm9udGFsaXphdGlvbkVycm9yKCJmaXRfZmFpbGVkIiwgZiJDb3VsZCBub3QgYnVpbGQgZnJvbnRhbCB3YXJwOiB7ZXhjfSIpIGZyb20gZXhjCiAgICAgICAgbWFwX3gsbWFwX3k9bWFwcGluZ1suLi4sMF0sbWFwcGluZ1suLi4sMV0KICAgICAgICB2YWxpZF9tYXA9KG1hcF94Pj0wKSYobWFwX3k+PTApJihtYXBfeDx3aWR0aCkmKG1hcF95PGhlaWdodCkKICAgICAgICBpZiB2YWxpZF9tYXAubWVhbigpPC4wMToKICAgICAgICAgICAgcmFpc2UgRnJvbnRhbGl6YXRpb25FcnJvcigiZml0X2ZhaWxlZCIsICJGcm9udGFsIHdhcnAgaXMgZGVnZW5lcmF0ZSIpCiAgICAgICAgc291cmNlX2ltZz1ucC5hc2FycmF5KGltYWdlLmNvbnZlcnQoIlJHQiIpKQogICAgICAgIHdhcnBlZD1jdjIucmVtYXAoc291cmNlX2ltZyxtYXBfeCxtYXBfeSxjdjIuSU5URVJfTElORUFSLGJvcmRlck1vZGU9Y3YyLkJPUkRFUl9SRUZMRUNUXzEwMSkKICAgICAgICBtYXNrPXZhbGlkX21hcC5hc3R5cGUobnAudWludDgpKjI1NQogICAgICAgIHNpZ21hPW1heCgxLHJvdW5kKG1pbih3aWR0aCxoZWlnaHQpKi4wMykpCiAgICAgICAgbWFzaz1jdjIuR2F1c3NpYW5CbHVyKG1hc2ssKDAsMCksc2lnbWFYPXNpZ21hLHNpZ21hWT1zaWdtYSkuYXN0eXBlKG5wLmZsb2F0MzIpLzI1NQogICAgICAgIG91dD1ucC5hc2FycmF5KGltYWdlLmNvbnZlcnQoIlJHQiIpLGR0eXBlPW5wLmZsb2F0MzIpKigxLW1hc2tbLi4uLE5vbmVdKSt3YXJwZWQuYXN0eXBlKG5wLmZsb2F0MzIpKm1hc2tbLi4uLE5vbmVdCiAgICAgICAgcmVzdWx0PUltYWdlLmZyb21hcnJheShucC5jbGlwKG91dCwwLDI1NSkuYXN0eXBlKG5wLnVpbnQ4KSwiUkdCIikKICAgICAgICByZXR1cm4gRnJvbnRhbGl6YXRpb25SZXN1bHQocmVzdWx0LCpwb3NlLG9jY2x1ZGVkLChwZXJmX2NvdW50ZXIoKS1zdGFydGVkKSoxMDAwKQoKICAgIGRlZiBfbWlycm9yX3BhaXJzKHNlbGYpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgY2FjaGVkPWdldGF0dHIoc2VsZiwiX21pcnJvcl9jYWNoZSIsTm9uZSkKICAgICAgICBpZiBjYWNoZWQgaXMgbm90IE5vbmU6cmV0dXJuIGNhY2hlZAogICAgICAgIG1lYW49c2VsZi51LnJlc2hhcGUoMywtMSxvcmRlcj0iRiIpLlQKICAgICAgICBmcm9tIHNjaXB5LnNwYXRpYWwgaW1wb3J0IGNLRFRyZWUKICAgICAgICBtaXJyb3JlZD1tZWFuLmNvcHkoKTsgbWlycm9yZWRbOiwwXSo9LTEKICAgICAgICBwYWlycz1jS0RUcmVlKG1lYW4pLnF1ZXJ5KG1pcnJvcmVkLGs9MSlbMV0uYXN0eXBlKG5wLmludDMyKQogICAgICAgIHNlbGYuX21pcnJvcl9jYWNoZT1wYWlycwogICAgICAgIHJldHVybiBwYWlycwoKCl9TSU5HTEVUT046IFREREZBVjJGcm9udGFsaXplciB8IE5vbmUgPSBOb25lCl9TSU5HTEVUT05fTE9DSz1Mb2NrKCkKZGVmIGdldF90ZGRmYV92Ml9mcm9udGFsaXplcigpIC0+IFREREZBVjJGcm9udGFsaXplcjoKICAgIGdsb2JhbCBfU0lOR0xFVE9OCiAgICB3aXRoIF9TSU5HTEVUT05fTE9DSzoKICAgICAgICBpZiBfU0lOR0xFVE9OIGlzIE5vbmU6X1NJTkdMRVRPTj1URERGQVYyRnJvbnRhbGl6ZXIoKQogICAgICAgIHJldHVybiBfU0lOR0xFVE9OCg=='))
FRONTALIZE_SCRIPTS = WORKDIR / 'frontalize_scripts'
FRONTALIZE_SCRIPTS.mkdir(parents=True, exist_ok=True)
(FRONTALIZE_SCRIPTS / 'run_frontalize_worker.py').write_bytes(base64.b64decode('ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgYXJncGFyc2UNCmltcG9ydCBjc3YNCmltcG9ydCBpbw0KaW1wb3J0IGpzb24NCmltcG9ydCBvcw0KaW1wb3J0IHN5cw0KaW1wb3J0IHRpbWUNCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aA0KZnJvbSB0eXBpbmcgaW1wb3J0IEl0ZXJhYmxlDQoNCmZyb20gUElMIGltcG9ydCBJbWFnZQ0KDQoNCkNMQVNTRVMgPSAoInN1cnByaXNlIiwgImZlYXIiLCAiZGlzZ3VzdCIsICJoYXBwaW5lc3MiLCAic2FkbmVzcyIsICJhbmdlciIsICJuZXV0cmFsIikNCk1PREVMUyA9ICgicG9zdGVyX3JhZmRiIiwgIm1hM2RfcmFmZGIiKQ0KREVGQVVMVF9BTkdMRVMgPSAoMCwgNDUsIDkwKQ0KDQoNCmRlZiBpbWFnZV9ieXRlcyhpbWFnZTogSW1hZ2UuSW1hZ2UpIC0+IGJ5dGVzOg0KICAgIGJ1ZmZlciA9IGlvLkJ5dGVzSU8oKQ0KICAgIGltYWdlLnNhdmUoYnVmZmVyLCBmb3JtYXQ9IlBORyIpDQogICAgcmV0dXJuIGJ1ZmZlci5nZXR2YWx1ZSgpDQoNCg0KZGVmIHJvdGF0ZShpbWFnZTogSW1hZ2UuSW1hZ2UsIGFuZ2xlOiBpbnQpIC0+IEltYWdlLkltYWdlOg0KICAgICIiIk1hdGNoIHRlc3RzL2JlbmNoX2Zyb250YWxpemF0aW9uX3JhZmRiLnB5Ojpfcm90YXRlIGJ5dGUtZm9yLWJ5dGUgaW4gYmVoYXZpb3IuIiIiDQogICAgaWYgYW5nbGUgPT0gMDoNCiAgICAgICAgcmV0dXJuIGltYWdlDQogICAgbWVhbiA9IHR1cGxlKGludCh2YWx1ZSkgZm9yIHZhbHVlIGluIGltYWdlLnJlc2l6ZSgoMSwgMSksIEltYWdlLkJJTElORUFSKS5nZXRwaXhlbCgoMCwgMCkpKQ0KICAgIHJldHVybiBpbWFnZS5yb3RhdGUoYW5nbGUsIHJlc2FtcGxlPUltYWdlLkJJQ1VCSUMsIGV4cGFuZD1UcnVlLCBmaWxsY29sb3I9bWVhbikNCg0KDQpkZWYgcm90YXRlX2FueShpbWFnZTogSW1hZ2UuSW1hZ2UsIGFuZ2xlOiBmbG9hdCkgLT4gSW1hZ2UuSW1hZ2U6DQogICAgIiIiU2FtZSByb3RhdGlvbi9maWxsIGFzIGByb3RhdGVgLCBmb3IgYSBmbG9hdCBhbmdsZSAoQURSLTAyOCByb2xsIGNvcnJlY3Rpb24pLiIiIg0KICAgIGlmIGFicyhhbmdsZSkgPCAxZS02Og0KICAgICAgICByZXR1cm4gaW1hZ2UNCiAgICBtZWFuID0gdHVwbGUoaW50KHZhbHVlKSBmb3IgdmFsdWUgaW4gaW1hZ2UucmVzaXplKCgxLCAxKSwgSW1hZ2UuQklMSU5FQVIpLmdldHBpeGVsKCgwLCAwKSkpDQogICAgcmV0dXJuIGltYWdlLnJvdGF0ZShhbmdsZSwgcmVzYW1wbGU9SW1hZ2UuQklDVUJJQywgZXhwYW5kPVRydWUsIGZpbGxjb2xvcj1tZWFuKQ0KDQoNCmRlZiBwcmVkaWN0aW9ucyhyZXN1bHQ6IGRpY3QgfCBOb25lKSAtPiBkaWN0Og0KICAgIG91dHB1dCA9IHt9DQogICAgZm9yIGZhY2UgaW4gKHJlc3VsdCBvciB7fSkuZ2V0KCJmYWNlcyIpIG9yIFtdOg0KICAgICAgICBmb3IgcHJlZGljdGlvbiBpbiBmYWNlLmdldCgicHJlZGljdGlvbnMiKSBvciBbXToNCiAgICAgICAgICAgIG1vZGVsX2lkID0gcHJlZGljdGlvbi5nZXQoIm1vZGVsX2lkIikNCiAgICAgICAgICAgIGlmIG1vZGVsX2lkIG5vdCBpbiBNT0RFTFMgb3IgbW9kZWxfaWQgaW4gb3V0cHV0Og0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICBpZiBwcmVkaWN0aW9uLmdldCgic3RhdHVzIikgPT0gIm9rIjoNCiAgICAgICAgICAgICAgICBsYWJlbHMgPSBwcmVkaWN0aW9uLmdldCgibGFiZWxzIikgb3IgW10NCiAgICAgICAgICAgICAgICBwcm9icyA9IHByZWRpY3Rpb24uZ2V0KCJwcm9icyIpIG9yIFtdDQogICAgICAgICAgICAgICAgdG9wID0gcHJlZGljdGlvbi5nZXQoInRvcDEiKSBvciB7fQ0KICAgICAgICAgICAgICAgIGlmIGxlbihsYWJlbHMpICE9IGxlbihwcm9icykgb3Igbm90IGxhYmVsczoNCiAgICAgICAgICAgICAgICAgICAgb3V0cHV0W21vZGVsX2lkXSA9IHsiZXJyb3IiOiAiaW52YWxpZF9vdXRwdXQifQ0KICAgICAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgICAgIG91dHB1dFttb2RlbF9pZF0gPSB7DQogICAgICAgICAgICAgICAgICAgICAgICAibGFiZWwiOiBzdHIodG9wLmdldCgibGFiZWwiLCBsYWJlbHNbbWF4KHJhbmdlKGxlbihwcm9icykpLCBrZXk9bGFtYmRhIGk6IHByb2JzW2ldKV0pKSwNCiAgICAgICAgICAgICAgICAgICAgICAgICJwcm9icyI6IHtzdHIobGFiZWwpOiByb3VuZChmbG9hdChwcm9iKSwgNSkgZm9yIGxhYmVsLCBwcm9iIGluIHppcChsYWJlbHMsIHByb2JzKX0sDQogICAgICAgICAgICAgICAgICAgIH0NCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgZXJyb3IgPSBwcmVkaWN0aW9uLmdldCgiZXJyb3IiKSBvciB7fQ0KICAgICAgICAgICAgICAgIG91dHB1dFttb2RlbF9pZF0gPSB7ImVycm9yIjogc3RyKGVycm9yLmdldCgiY29kZSIpIG9yIHByZWRpY3Rpb24uZ2V0KCJzdGF0dXMiKSBvciAicHJlZGljdGlvbl9lcnJvciIpLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIm1lc3NhZ2UiOiBzdHIoZXJyb3IuZ2V0KCJtZXNzYWdlIikgb3IgIiIpWzozMDBdfQ0KICAgIHRvcF9lcnJvciA9IChyZXN1bHQgb3Ige30pLmdldCgiZXJyb3IiKSBvciB7fQ0KICAgIGZhbGxiYWNrID0geyJlcnJvciI6IHN0cih0b3BfZXJyb3IuZ2V0KCJjb2RlIikgb3IgIm5vX3ByZWRpY3Rpb24iKSwgIm1lc3NhZ2UiOiBzdHIodG9wX2Vycm9yLmdldCgibWVzc2FnZSIpIG9yICIiKVs6MzAwXX0NCiAgICBmb3IgbW9kZWxfaWQgaW4gTU9ERUxTOg0KICAgICAgICBvdXRwdXQuc2V0ZGVmYXVsdChtb2RlbF9pZCwgZGljdChmYWxsYmFjaykpDQogICAgcmV0dXJuIG91dHB1dA0KDQoNCmRlZiBwYXJzZV9hbmdsZXModmFsdWU6IHN0ciB8IEl0ZXJhYmxlW2ludF0pIC0+IGxpc3RbaW50XToNCiAgICB0cnk6DQogICAgICAgIHZhbHVlcyA9IFtpbnQocGFydC5zdHJpcCgpKSBmb3IgcGFydCBpbiB2YWx1ZS5zcGxpdCgiLCIpIGlmIHBhcnQuc3RyaXAoKV0gaWYgaXNpbnN0YW5jZSh2YWx1ZSwgc3RyKSBlbHNlIFtpbnQoeCkgZm9yIHggaW4gdmFsdWVdDQogICAgZXhjZXB0IChUeXBlRXJyb3IsIFZhbHVlRXJyb3IpIGFzIGV4YzoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYW5nbGVzIG11c3QgYmUgY29tbWEtc2VwYXJhdGVkIGludGVnZXJzIikgZnJvbSBleGMNCiAgICBpZiBub3QgdmFsdWVzIG9yIGxlbih2YWx1ZXMpICE9IGxlbihzZXQodmFsdWVzKSk6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImFuZ2xlcyBtdXN0IGNvbnRhaW4gYXQgbGVhc3Qgb25lIHVuaXF1ZSBpbnRlZ2VyIikNCiAgICByZXR1cm4gdmFsdWVzDQoNCg0KZGVmIG1haW4oKSAtPiBpbnQ6DQogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IkJlbmNobWFyayByYXcgYW5kIDNEREZBLVYyIGZyb250YWxpemVkIFJBRi1EQiBpbWFnZXMuIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJ1bmRsZS1yb290IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcHJvamVjdC1yb290IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZW1vY2EtdXJsIiwgZGVmYXVsdD1Ob25lKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0LWpzb25sIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2hhcmQtaW5kZXgiLCB0eXBlPWludCwgcmVxdWlyZWQ9VHJ1ZSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNoYXJkcyIsIHR5cGU9aW50LCBkZWZhdWx0PTIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1saW1pdCIsIHR5cGU9aW50LCBkZWZhdWx0PTAsIGhlbHA9IkxpbWl0IHNvcnRlZCBpbWFnZXMgYmVmb3JlIHNoYXJkaW5nOyAwIG1lYW5zIGFsbC4iKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYW5nbGVzIiwgZGVmYXVsdD0iMCw0NSw5MCIsIGhlbHA9IkNvbW1hLXNlcGFyYXRlZCBpbi1wbGFuZSBhbmdsZXMgKGRlZmF1bHQ6IDAsNDUsOTApLiIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tb2RlIiwgY2hvaWNlcz0oImZyb250YWwzZCIsICJyb2xsMmQiKSwgZGVmYXVsdD0iZnJvbnRhbDNkIiwNCiAgICAgICAgICAgICAgICAgICAgICAgIGhlbHA9ImZyb250YWwzZDogQURSLTAyNiAzRCB3YXJwOyByb2xsMmQ6IEFEUi0wMjggcm9sbCBlc3RpbWF0ZSArIDJEIHJvdGF0aW9uIG9ubHkuIikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1pbi1yb2xsIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD01LjAsDQogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJyb2xsMmQ6IGJlbG93IHRoaXMgfHJvbGx8IHRoZSBjb3JyZWN0ZWQgaW1hZ2UgZXF1YWxzIHRoZSBpbnB1dCwgc28gdGhlIHJhdyAiDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICJyZXN1bHQgaXMgcmV1c2VkIGluc3RlYWQgb2YgYSBzZWNvbmQgYW5hbHl6ZSAodGhyZXNob2xkcyBhcmUgYXBwbGllZCBwb3N0IGhvYykuIikNCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQ0KICAgIGlmIGFyZ3Muc2hhcmRzIDwgMSBvciBub3QgMCA8PSBhcmdzLnNoYXJkX2luZGV4IDwgYXJncy5zaGFyZHM6DQogICAgICAgIHBhcnNlci5lcnJvcigicmVxdWlyZSAtLXNoYXJkcyA+PSAxIGFuZCAwIDw9IC0tc2hhcmQtaW5kZXggPCAtLXNoYXJkcyIpDQogICAgaWYgYXJncy5saW1pdCA8IDA6DQogICAgICAgIHBhcnNlci5lcnJvcigiLS1saW1pdCBjYW5ub3QgYmUgbmVnYXRpdmUiKQ0KICAgIHRyeToNCiAgICAgICAgYW5nbGVzID0gcGFyc2VfYW5nbGVzKGFyZ3MuYW5nbGVzKQ0KICAgIGV4Y2VwdCBWYWx1ZUVycm9yIGFzIGV4YzoNCiAgICAgICAgcGFyc2VyLmVycm9yKHN0cihleGMpKQ0KDQogICAgYnVuZGxlID0gYXJncy5idW5kbGVfcm9vdC5yZXNvbHZlKCkNCiAgICBwcm9qZWN0ID0gYXJncy5wcm9qZWN0X3Jvb3QucmVzb2x2ZSgpDQogICAgc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihwcm9qZWN0KSkNCiAgICBvcy5lbnZpcm9uWyJNQTNEX0RFVklDRSJdID0gImN1ZGE6MCIgaWYgb3MuZW52aXJvbi5nZXQoIkNVREFfVklTSUJMRV9ERVZJQ0VTIikgaXMgbm90IE5vbmUgZWxzZSAiYXV0byINCiAgICBmcm9tIHNlcnZpY2VzLmZyb250YWxpemF0aW9uIGltcG9ydCBnZXRfdGRkZmFfdjJfZnJvbnRhbGl6ZXINCiAgICBmcm9tIHdlYi5vcmNoZXN0cmF0b3IgaW1wb3J0IGFuYWx5emUsIGNoZWNrX2Vtb2NhDQogICAgaW1wb3J0IGluc3BlY3QNCiAgICAjIFRoZSBLYWdnbGUgRGF0YXNldCBzbmFwc2hvdCAoMjAyNi0wOS0zMCkgcHJlZGF0ZXMgTTcncyBgbWF4X2ZhY2VzYCBhcmd1bWVudDsgdGhhdA0KICAgICMgdmVyc2lvbiBpcyBzaW5nbGUtZmFjZSBieSBkZWZhdWx0LCBzbyBvbmx5IHBhc3MgaXQgd2hlbiB0aGUgc3RhZ2VkIGNvZGUgYWNjZXB0cyBpdC4NCiAgICBhbmFseXplX2V4dHJhID0geyJtYXhfZmFjZXMiOiAxfSBpZiAibWF4X2ZhY2VzIiBpbiBpbnNwZWN0LnNpZ25hdHVyZShhbmFseXplKS5wYXJhbWV0ZXJzIGVsc2Uge30NCg0KICAgIGxhYmVsc19wYXRoID0gYnVuZGxlIC8gImRhdGFzZXQiIC8gInRlc3RfbGFiZWxzLmNzdiINCiAgICB3aXRoIGxhYmVsc19wYXRoLm9wZW4oZW5jb2Rpbmc9InV0Zi04LXNpZyIsIG5ld2xpbmU9IiIpIGFzIHN0cmVhbToNCiAgICAgICAgbGFiZWxzID0ge3N0cihyb3dbImltYWdlIl0pOiBpbnQocm93WyJsYWJlbCJdKSBmb3Igcm93IGluIGNzdi5EaWN0UmVhZGVyKHN0cmVhbSl9DQogICAgIyBDYW5vbmljYWxpemUgdGhlIGtleSB3aXRoIGl0cyBSQUYtREIgY2xhc3MgZGlyZWN0b3J5LCBtYXRjaGluZyB0aGUgbG9jYWwgcmVmZXJlbmNlDQogICAgIyAoYGsvZmlsZW5hbWUuanBnYCkgYW5kIHByZXZlbnRpbmcgYSBzYW1lLW5hbWVkIGZpbGUgaW4gYW5vdGhlciBjbGFzcyBmcm9tIGNvbGxpZGluZy4NCiAgICBlbnRyaWVzID0gc29ydGVkKCgoZiJ7bGFiZWx9L3tQYXRoKGltYWdlKS5uYW1lfSIsIGxhYmVsKSBmb3IgaW1hZ2UsIGxhYmVsIGluIGxhYmVscy5pdGVtcygpKSwNCiAgICAgICAgICAgICAgICAgICAgIGtleT1sYW1iZGEgaXRlbTogKGl0ZW1bMV0sIGl0ZW1bMF0pKQ0KICAgIGlmIGFyZ3MubGltaXQ6DQogICAgICAgIGVudHJpZXMgPSBlbnRyaWVzWzphcmdzLmxpbWl0XQ0KICAgIGVudHJpZXMgPSBbZW50cnkgZm9yIGluZGV4LCBlbnRyeSBpbiBlbnVtZXJhdGUoZW50cmllcykgaWYgaW5kZXggJSBhcmdzLnNoYXJkcyA9PSBhcmdzLnNoYXJkX2luZGV4XQ0KDQogICAgZW1vY2EgPSB7Im1vZGUiOiAicmVhbCIsICJ1cmwiOiBhcmdzLmVtb2NhX3VybCBvciBvcy5lbnZpcm9uLmdldCgiRU1PQ0FfVVJMIiwgImh0dHA6Ly8xMjcuMC4wLjE6ODc2NSIpLCAidGltZW91dF9zIjogMTIwfQ0KICAgIGhlYWx0aCA9IGNoZWNrX2Vtb2NhKGVtb2NhKQ0KICAgIGlmIG5vdCBoZWFsdGguZ2V0KCJvayIpOg0KICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJFTU9DQSBpcyBub3QgaGVhbHRoeTsgcmVmdXNpbmcgbW9jayBmYWxsYmFjazoge2hlYWx0aH0iKQ0KICAgIGZyb250YWxpemVyID0gZ2V0X3RkZGZhX3YyX2Zyb250YWxpemVyKCkNCg0KICAgIG91dF9wYXRoID0gYXJncy5vdXRfanNvbmwucmVzb2x2ZSgpDQogICAgb3V0X3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICBkb25lOiBzZXRbdHVwbGVbc3RyLCBpbnRdXSA9IHNldCgpDQogICAgaWYgb3V0X3BhdGguaXNfZmlsZSgpOg0KICAgICAgICB3aXRoIG91dF9wYXRoLm9wZW4oZW5jb2Rpbmc9InV0Zi04IikgYXMgc3RyZWFtOg0KICAgICAgICAgICAgZm9yIGxpbmUgaW4gc3RyZWFtOg0KICAgICAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICAgICAgcm93ID0ganNvbi5sb2FkcyhsaW5lKQ0KICAgICAgICAgICAgICAgICAgICBkb25lLmFkZCgoc3RyKHJvd1siaW1hZ2UiXSksIGludChyb3dbImFuZ2xlIl0pKSkNCiAgICAgICAgICAgICAgICBleGNlcHQgKGpzb24uSlNPTkRlY29kZUVycm9yLCBLZXlFcnJvciwgVHlwZUVycm9yLCBWYWx1ZUVycm9yKToNCiAgICAgICAgICAgICAgICAgICAgY29udGludWUNCiAgICB0b2RvID0gWyhpbWFnZSwgbGFiZWwsIGFuZ2xlKSBmb3IgaW1hZ2UsIGxhYmVsIGluIGVudHJpZXMgZm9yIGFuZ2xlIGluIGFuZ2xlcyBpZiAoaW1hZ2UsIGFuZ2xlKSBub3QgaW4gZG9uZV0NCiAgICBwcmludChmIlNoYXJkIHthcmdzLnNoYXJkX2luZGV4fS97YXJncy5zaGFyZHN9OiBpbWFnZXM9e2xlbihlbnRyaWVzKX0gcm93c19wZW5kaW5nPXtsZW4odG9kbyl9IGFuZ2xlcz17YW5nbGVzfSIsIGZsdXNoPVRydWUpDQoNCiAgICBkZWYgaW5mZXIoaW1hZ2U6IEltYWdlLkltYWdlKSAtPiB0dXBsZVtkaWN0LCBpbnQsIGludF06DQogICAgICAgIHJldHJpZXMgPSAwDQogICAgICAgIHN0YXJ0ZWQgPSB0aW1lLnBlcmZfY291bnRlcigpDQogICAgICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDMpOg0KICAgICAgICAgICAgcmVzdWx0ID0gYW5hbHl6ZShpbWFnZV9ieXRlcyhpbWFnZSksIGVtb2NhPWVtb2NhLCBlbmdpbmVfbW9kZT0icmVhbCIsICoqYW5hbHl6ZV9leHRyYSkNCiAgICAgICAgICAgIHBhcnNlZCA9IHByZWRpY3Rpb25zKHJlc3VsdCkNCiAgICAgICAgICAgIHVuYXZhaWxhYmxlID0gInNlcnZpY2VfdW5hdmFpbGFibGUiIGluIHN0cihwYXJzZWQuZ2V0KCJtYTNkX3JhZmRiIiwge30pLmdldCgiZXJyb3IiLCAiIikpDQogICAgICAgICAgICBpZiBub3QgdW5hdmFpbGFibGUgb3IgYXR0ZW1wdCA9PSAyOg0KICAgICAgICAgICAgICAgIHJldHVybiBwYXJzZWQsIHJvdW5kKCh0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZCkgKiAxMDAwKSwgcmV0cmllcw0KICAgICAgICAgICAgcmV0cmllcyArPSAxDQogICAgICAgICAgICB0aW1lLnNsZWVwKDUpDQogICAgICAgIHJhaXNlIEFzc2VydGlvbkVycm9yKCJ1bnJlYWNoYWJsZSIpDQoNCiAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQ0KICAgIHdpdGggb3V0X3BhdGgub3BlbigiYSIsIGVuY29kaW5nPSJ1dGYtOCIsIGJ1ZmZlcmluZz0xKSBhcyBzdHJlYW06DQogICAgICAgIGZvciBsb2NhbF9pbmRleCwgKHJlbGF0aXZlLCBsYWJlbF9pbmRleCwgYW5nbGUpIGluIGVudW1lcmF0ZSh0b2RvLCAxKToNCiAgICAgICAgICAgIHJvdyA9IHsiaW1hZ2UiOiByZWxhdGl2ZSwgImxhYmVsIjogQ0xBU1NFU1tsYWJlbF9pbmRleCAtIDFdLCAiYW5nbGUiOiBhbmdsZSwgIm1vZGUiOiBhcmdzLm1vZGUsDQogICAgICAgICAgICAgICAgICAgInBvc2UiOiBOb25lLCAib2NjbHVkZWQiOiBOb25lLA0KICAgICAgICAgICAgICAgICAgICJ0aW1pbmdzIjogeyJtc19yYXciOiBOb25lLCAibXNfZnJvbnRhbGl6ZSI6IE5vbmUsICJtc19mcm9udGFsIjogTm9uZX0sDQogICAgICAgICAgICAgICAgICAgInJhdyI6IHt9LCAiZnJvbnRhbCI6IHt9LCAiZW1vY2FfcmV0cmllcyI6IHsicmF3IjogMCwgImZyb250YWwiOiAwfX0NCiAgICAgICAgICAgIGltYWdlX3BhdGggPSBidW5kbGUgLyAiZGF0YXNldCIgLyAiREFUQVNFVCIgLyAidGVzdCIgLyByZWxhdGl2ZQ0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIHdpdGggSW1hZ2Uub3BlbihpbWFnZV9wYXRoKSBhcyBzb3VyY2U6DQogICAgICAgICAgICAgICAgICAgIHJvdGF0ZWQgPSByb3RhdGUoc291cmNlLmNvbnZlcnQoIlJHQiIpLCBhbmdsZSkNCiAgICAgICAgICAgICAgICByYXdfc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkNCiAgICAgICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgICAgIHJvd1sicmF3Il0sIHJvd1sidGltaW5ncyJdWyJtc19yYXciXSwgcm93WyJlbW9jYV9yZXRyaWVzIl1bInJhdyJdID0gaW5mZXIocm90YXRlZCkNCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoNCiAgICAgICAgICAgICAgICAgICAgcm93WyJ0aW1pbmdzIl1bIm1zX3JhdyJdID0gcm91bmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSByYXdfc3RhcnRlZCkgKiAxMDAwKQ0KICAgICAgICAgICAgICAgICAgICByb3dbInJhdyJdID0ge21pZDogeyJlcnJvciI6ICJhbmFseXplX2ZhaWxlZCJ9IGZvciBtaWQgaW4gTU9ERUxTfQ0KICAgICAgICAgICAgICAgICAgICByb3dbInJhd19lcnJvciJdID0gZiJ7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30iDQogICAgICAgICAgICAgICAgZnJvbnRhbF9zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQ0KICAgICAgICAgICAgICAgIGlmIGFyZ3MubW9kZSA9PSAicm9sbDJkIjoNCiAgICAgICAgICAgICAgICAgICAgIyBBRFItMDI4OiBlc3RpbWF0ZSByb2xsIHdpdGggM0RERkEtVjIgYW5kIHVuZG8gaXQgd2l0aCBhIHBsYWluIDJEIHJvdGF0aW9uLg0KICAgICAgICAgICAgICAgICAgICAjIFNpZ24gbWVhc3VyZWQgb24gUkFGLURCIGNyb3BzOiByb3RhdGUoK2EpIC0+IHJvbGwgfiArYSwgc28gcm90YXRlKC1yb2xsKSBmaXhlcyBpdC4NCiAgICAgICAgICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgICAgICAgICAgXywgcG9zZSA9IGZyb250YWxpemVyLmVzdGltYXRlKHJvdGF0ZWQpDQogICAgICAgICAgICAgICAgICAgICAgICByb2xsID0gZmxvYXQocG9zZVsyXSkNCiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sicG9zZSJdID0gW3JvdW5kKGZsb2F0KHBvc2VbMF0pLCAyKSwgcm91bmQoZmxvYXQocG9zZVsxXSksIDIpLCByb3VuZChyb2xsLCAyKV0NCiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sidGltaW5ncyJdWyJtc19mcm9udGFsaXplIl0gPSByb3VuZCgodGltZS5wZXJmX2NvdW50ZXIoKSAtIGZyb250YWxfc3RhcnRlZCkgKiAxMDAwKQ0KICAgICAgICAgICAgICAgICAgICAgICAgaWYgYWJzKHJvbGwpIDwgYXJncy5taW5fcm9sbDoNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICByb3dbInJvbGxfYXBwbGllZF9kZWciXSA9IDAuMA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvd1siZnJvbnRhbCJdID0ganNvbi5sb2Fkcyhqc29uLmR1bXBzKHJvd1sicmF3Il0pKQ0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvd1siZnJvbnRhbF9yZXVzZWRfcmF3Il0gPSBUcnVlDQogICAgICAgICAgICAgICAgICAgICAgICBlbHNlOg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sicm9sbF9hcHBsaWVkX2RlZyJdID0gcm91bmQoLXJvbGwsIDIpDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgY29ycmVjdGVkID0gcm90YXRlX2FueShyb3RhdGVkLCAtcm9sbCkNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBpbmZlcl9zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQ0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcm93WyJmcm9udGFsIl0sIHJvd1sidGltaW5ncyJdWyJtc19mcm9udGFsIl0sIHJvd1siZW1vY2FfcmV0cmllcyJdWyJmcm9udGFsIl0gPSBpbmZlcihjb3JyZWN0ZWQpDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6DQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sidGltaW5ncyJdWyJtc19mcm9udGFsIl0gPSByb3VuZCgodGltZS5wZXJmX2NvdW50ZXIoKSAtIGluZmVyX3N0YXJ0ZWQpICogMTAwMCkNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcm93WyJmcm9udGFsIl0gPSB7bWlkOiB7ImVycm9yIjogImFuYWx5emVfZmFpbGVkIn0gZm9yIG1pZCBpbiBNT0RFTFN9DQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvd1siZnJvbnRhbF9hbmFseXplX2Vycm9yIl0gPSBmInt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSINCiAgICAgICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6DQogICAgICAgICAgICAgICAgICAgICAgICByb3dbInRpbWluZ3MiXVsibXNfZnJvbnRhbGl6ZSJdID0gcm91bmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSBmcm9udGFsX3N0YXJ0ZWQpICogMTAwMCkNCiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1siZnJvbnRhbF9lcnJvciJdID0gZiJ7dHlwZShleGMpLl9fbmFtZV9ffToge2dldGF0dHIoZXhjLCAnbWVzc2FnZScsIGV4Yyl9Ig0KICAgICAgICAgICAgICAgICAgICAgICAgcm93WyJmcm9udGFsIl0gPSB7bWlkOiB7ImVycm9yIjogInJvbGxfZXN0aW1hdGVfZmFpbGVkIn0gZm9yIG1pZCBpbiBNT0RFTFN9DQogICAgICAgICAgICAgICAgZWxzZToNCiAgICAgICAgICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0ID0gZnJvbnRhbGl6ZXIuZnJvbnRhbGl6ZShyb3RhdGVkKQ0KICAgICAgICAgICAgICAgICAgICAgICAgcm93WyJ0aW1pbmdzIl1bIm1zX2Zyb250YWxpemUiXSA9IHJvdW5kKHJlc3VsdC50aW1lX21zKQ0KICAgICAgICAgICAgICAgICAgICAgICAgcm93WyJwb3NlIl0gPSBbcm91bmQocmVzdWx0Lnlhd19kZWcsIDIpLCByb3VuZChyZXN1bHQucGl0Y2hfZGVnLCAyKSwgcm91bmQocmVzdWx0LnJvbGxfZGVnLCAyKV0NCiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sib2NjbHVkZWQiXSA9IHJvdW5kKHJlc3VsdC5vY2NsdWRlZF9mcmFjdGlvbiwgNCkNCiAgICAgICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6DQogICAgICAgICAgICAgICAgICAgICAgICByb3dbInRpbWluZ3MiXVsibXNfZnJvbnRhbGl6ZSJdID0gcm91bmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSBmcm9udGFsX3N0YXJ0ZWQpICogMTAwMCkNCiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1siZnJvbnRhbF9lcnJvciJdID0gZiJ7dHlwZShleGMpLl9fbmFtZV9ffToge2dldGF0dHIoZXhjLCAnbWVzc2FnZScsIGV4Yyl9Ig0KICAgICAgICAgICAgICAgICAgICAgICAgcm93WyJmcm9udGFsIl0gPSB7bWlkOiB7ImVycm9yIjogImZyb250YWxpemVfZmFpbGVkIn0gZm9yIG1pZCBpbiBNT0RFTFN9DQogICAgICAgICAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgICAgICAgICBpbmZlcl9zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQ0KICAgICAgICAgICAgICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvd1siZnJvbnRhbCJdLCByb3dbInRpbWluZ3MiXVsibXNfZnJvbnRhbCJdLCByb3dbImVtb2NhX3JldHJpZXMiXVsiZnJvbnRhbCJdID0gaW5mZXIocmVzdWx0LmltYWdlKQ0KICAgICAgICAgICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6DQogICAgICAgICAgICAgICAgICAgICAgICAgICAgcm93WyJ0aW1pbmdzIl1bIm1zX2Zyb250YWwiXSA9IHJvdW5kKCh0aW1lLnBlcmZfY291bnRlcigpIC0gaW5mZXJfc3RhcnRlZCkgKiAxMDAwKQ0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvd1siZnJvbnRhbCJdID0ge21pZDogeyJlcnJvciI6ICJhbmFseXplX2ZhaWxlZCJ9IGZvciBtaWQgaW4gTU9ERUxTfQ0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvd1siZnJvbnRhbF9hbmFseXplX2Vycm9yIl0gPSBmInt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSINCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOg0KICAgICAgICAgICAgICAgIHJvd1siaW5wdXRfZXJyb3IiXSA9IGYie3R5cGUoZXhjKS5fX25hbWVfX306IHtleGN9Ig0KICAgICAgICAgICAgICAgIHJvd1sicmF3Il0gPSB7bWlkOiB7ImVycm9yIjogImlucHV0X2ZhaWxlZCJ9IGZvciBtaWQgaW4gTU9ERUxTfQ0KICAgICAgICAgICAgICAgIHJvd1siZnJvbnRhbCJdID0ge21pZDogeyJlcnJvciI6ICJpbnB1dF9mYWlsZWQifSBmb3IgbWlkIGluIE1PREVMU30NCiAgICAgICAgICAgIHN0cmVhbS53cml0ZShqc29uLmR1bXBzKHJvdywgZW5zdXJlX2FzY2lpPUZhbHNlKSArICJcbiIpDQogICAgICAgICAgICBzdHJlYW0uZmx1c2goKQ0KICAgICAgICAgICAgb3MuZnN5bmMoc3RyZWFtLmZpbGVubygpKQ0KICAgICAgICAgICAgaWYgbG9jYWxfaW5kZXggJSAyNSA9PSAwIG9yIGxvY2FsX2luZGV4ID09IGxlbih0b2RvKToNCiAgICAgICAgICAgICAgICByYXRlID0gKHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkKSAvIGxvY2FsX2luZGV4DQogICAgICAgICAgICAgICAgIyBQcm9ncmVzcyBmaWxlIHJlYWRhYmxlIGZyb20gS2FnZ2xlJ3MgT3V0cHV0IHBhbmVsIHdoaWxlIHRoZSBjZWxsIGlzIHN0aWxsIHJ1bm5pbmcuDQogICAgICAgICAgICAgICAgcHJvZ3Jlc3MgPSB7InNoYXJkIjogYXJncy5zaGFyZF9pbmRleCwgImRvbmUiOiBsb2NhbF9pbmRleCwgInBlbmRpbmdfdG90YWwiOiBsZW4odG9kbyksDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgImFscmVhZHlfZG9uZV9iZWZvcmVfc3RhcnQiOiBsZW4oZG9uZSksICJwZXJjZW50Ijogcm91bmQoMTAwICogbG9jYWxfaW5kZXggLyBsZW4odG9kbyksIDIpLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICJzZWNvbmRzX3Blcl9yb3ciOiByb3VuZChyYXRlLCAzKSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZXRhX21pbnV0ZXMiOiByb3VuZChyYXRlICogKGxlbih0b2RvKSAtIGxvY2FsX2luZGV4KSAvIDYwLCAxKSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAidXBkYXRlZF91bml4Ijogcm91bmQodGltZS50aW1lKCkpfQ0KICAgICAgICAgICAgICAgIG91dF9wYXRoLndpdGhfbmFtZShmInByb2dyZXNzX3NoYXJke2FyZ3Muc2hhcmRfaW5kZXh9Lmpzb24iKS53cml0ZV90ZXh0KA0KICAgICAgICAgICAgICAgICAgICBqc29uLmR1bXBzKHByb2dyZXNzLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpDQogICAgICAgICAgICAgICAgcHJpbnQoZiJzaGFyZD17YXJncy5zaGFyZF9pbmRleH0gcHJvZ3Jlc3M9e2xvY2FsX2luZGV4fS97bGVuKHRvZG8pfSBpbWFnZT17cmVsYXRpdmV9IGFuZ2xlPXthbmdsZX0gcmF0ZT17cmF0ZTouMmZ9cy9yb3ciLCBmbHVzaD1UcnVlKQ0KICAgIHJldHVybiAwDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkNCg=='))
(FRONTALIZE_SCRIPTS / 'summarize_frontalize.py').write_bytes(base64.b64decode('ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHN5cwpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyLCBkZWZhdWx0ZGljdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpDTEFTU0VTID0gKCJzdXJwcmlzZSIsICJmZWFyIiwgImRpc2d1c3QiLCAiaGFwcGluZXNzIiwgInNhZG5lc3MiLCAiYW5nZXIiLCAibmV1dHJhbCIpCk1PREVMUyA9ICgicG9zdGVyX3JhZmRiIiwgIm1hM2RfcmFmZGIiKQpCUkFOQ0hFUyA9ICgicmF3IiwgImZyb250YWwiKQoKCmRlZiBsb2FkX3Jvd3MocmVzdWx0c19kaXI6IFBhdGgpIC0+IGxpc3RbZGljdF06CiAgICBsYXRlc3Q6IGRpY3RbdHVwbGVbc3RyLCBpbnRdLCBkaWN0XSA9IHt9CiAgICBmaWxlcyA9IHNvcnRlZChyZXN1bHRzX2Rpci5nbG9iKCJzaGFyZF8qLmpzb25sIikpCiAgICBpZiBub3QgZmlsZXM6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJObyBzaGFyZF8qLmpzb25sIGZpbGVzIGluIHtyZXN1bHRzX2Rpcn0iKQogICAgZm9yIHBhdGggaW4gZmlsZXM6CiAgICAgICAgd2l0aCBwYXRoLm9wZW4oZW5jb2Rpbmc9InV0Zi04IikgYXMgc3RyZWFtOgogICAgICAgICAgICBmb3IgbGluZSBpbiBzdHJlYW06CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcm93ID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgICAgICAgICAgICAgIGxhdGVzdFsoc3RyKHJvd1siaW1hZ2UiXSksIGludChyb3dbImFuZ2xlIl0pKV0gPSByb3cKICAgICAgICAgICAgICAgIGV4Y2VwdCAoanNvbi5KU09ORGVjb2RlRXJyb3IsIEtleUVycm9yLCBUeXBlRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICByZXR1cm4gc29ydGVkKGxhdGVzdC52YWx1ZXMoKSwga2V5PWxhbWJkYSByb3c6IChzdHIocm93WyJpbWFnZSJdKSwgaW50KHJvd1siYW5nbGUiXSkpKQoKCmRlZiBwcmVkaWN0aW9uKHJvdzogZGljdCwgYnJhbmNoOiBzdHIsIG1vZGVsOiBzdHIpIC0+IGRpY3QgfCBOb25lOgogICAgdmFsdWUgPSAocm93LmdldChicmFuY2gpIG9yIHt9KS5nZXQobW9kZWwpCiAgICBpZiBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgZGljdCkgb3IgImxhYmVsIiBub3QgaW4gdmFsdWU6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJldHVybiB2YWx1ZQoKCmRlZiBtZXRyaWNzKHJvd3M6IGxpc3RbZGljdF0sIGJyYW5jaDogc3RyLCBtb2RlbDogc3RyKSAtPiBkaWN0OgogICAgbiA9IGxlbihyb3dzKQogICAgY29uZnVzaW9uID0gW1swIGZvciBfIGluIENMQVNTRVNdIGZvciBfIGluIENMQVNTRVNdCiAgICB2YWxpZCA9IDAKICAgIGNvcnJlY3QgPSAwCiAgICB0cnVlX3Byb2JzID0gW10KICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICB0cnVlX2xhYmVsID0gc3RyKHJvdy5nZXQoImxhYmVsIiwgIiIpKQogICAgICAgIGlmIHRydWVfbGFiZWwgbm90IGluIENMQVNTRVM6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdHJ1ZV9pbmRleCA9IENMQVNTRVMuaW5kZXgodHJ1ZV9sYWJlbCkKICAgICAgICBwcmVkID0gcHJlZGljdGlvbihyb3csIGJyYW5jaCwgbW9kZWwpCiAgICAgICAgaWYgcHJlZCBpcyBOb25lIG9yIHByZWQuZ2V0KCJsYWJlbCIpIG5vdCBpbiBDTEFTU0VTOgogICAgICAgICAgICAjIE1pc3Npbmcgb3IgZmFpbGVkIHByZWRpY3Rpb25zIGNvdW50IGFzIHdyb25nIGFuZCBhcyBhIGZhbHNlIG5lZ2F0aXZlIGZvciB0cnV0aC4KICAgICAgICAgICAgY29udGludWUKICAgICAgICB2YWxpZCArPSAxCiAgICAgICAgcHJlZF9pbmRleCA9IENMQVNTRVMuaW5kZXgocHJlZFsibGFiZWwiXSkKICAgICAgICBjb25mdXNpb25bdHJ1ZV9pbmRleF1bcHJlZF9pbmRleF0gKz0gMQogICAgICAgIGNvcnJlY3QgKz0gaW50KHByZWRfaW5kZXggPT0gdHJ1ZV9pbmRleCkKICAgICAgICBwcm9icyA9IHByZWQuZ2V0KCJwcm9icyIpIG9yIHt9CiAgICAgICAgdHJ5OgogICAgICAgICAgICB0cnVlX3Byb2JzLmFwcGVuZChmbG9hdChwcm9icy5nZXQodHJ1ZV9sYWJlbCwgMC4wKSkpCiAgICAgICAgZXhjZXB0IChUeXBlRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgICAgICB0cnVlX3Byb2JzLmFwcGVuZCgwLjApCgogICAgcGVyX2NsYXNzID0ge30KICAgIGYxX3Njb3JlcyA9IFtdCiAgICBmb3IgaW5kZXgsIG5hbWUgaW4gZW51bWVyYXRlKENMQVNTRVMpOgogICAgICAgIHRwID0gY29uZnVzaW9uW2luZGV4XVtpbmRleF0KICAgICAgICBzdXBwb3J0ID0gc3VtKDEgZm9yIHJvdyBpbiByb3dzIGlmIHJvdy5nZXQoImxhYmVsIikgPT0gbmFtZSkKICAgICAgICBwcmVkaWN0ZWQgPSBzdW0oY29uZnVzaW9uW2FjdHVhbF1baW5kZXhdIGZvciBhY3R1YWwgaW4gcmFuZ2UobGVuKENMQVNTRVMpKSkKICAgICAgICAjIEZhaWxlZCBwcmVkaWN0aW9ucyBiZWxvbmcgaW4gYWN0dWFsIHN1cHBvcnQgYnV0IG5vdCBwcmVkaWN0ZWQgc3VwcG9ydC4KICAgICAgICBwcmVjaXNpb24gPSB0cCAvIHByZWRpY3RlZCBpZiBwcmVkaWN0ZWQgZWxzZSAwLjAKICAgICAgICByZWNhbGwgPSB0cCAvIHN1cHBvcnQgaWYgc3VwcG9ydCBlbHNlIDAuMAogICAgICAgIGYxID0gMiAqIHByZWNpc2lvbiAqIHJlY2FsbCAvIChwcmVjaXNpb24gKyByZWNhbGwpIGlmIHByZWNpc2lvbiArIHJlY2FsbCBlbHNlIDAuMAogICAgICAgIHBlcl9jbGFzc1tuYW1lXSA9IHsic3VwcG9ydCI6IHN1cHBvcnQsICJmMSI6IGYxLCAiZjFfcGN0IjogMTAwICogZjF9CiAgICAgICAgZjFfc2NvcmVzLmFwcGVuZChmMSkKICAgIHJldHVybiB7CiAgICAgICAgIm4iOiBuLAogICAgICAgICJuX3ZhbGlkIjogdmFsaWQsCiAgICAgICAgIm5fZmFpbGVkIjogbiAtIHZhbGlkLAogICAgICAgICJjb3ZlcmFnZSI6IHZhbGlkIC8gbiBpZiBuIGVsc2UgTm9uZSwKICAgICAgICAiYWNjdXJhY3lfdmFsaWQiOiBjb3JyZWN0IC8gdmFsaWQgaWYgdmFsaWQgZWxzZSBOb25lLAogICAgICAgICJhY2N1cmFjeV9hbGwiOiBjb3JyZWN0IC8gbiBpZiBuIGVsc2UgTm9uZSwKICAgICAgICAibWFjcm9fZjEiOiBzdW0oZjFfc2NvcmVzKSAvIGxlbihDTEFTU0VTKSBpZiBuIGVsc2UgTm9uZSwKICAgICAgICAibWVhbl9wX3RydWVfbGFiZWwiOiBzdW0odHJ1ZV9wcm9icykgLyBsZW4odHJ1ZV9wcm9icykgaWYgdHJ1ZV9wcm9icyBlbHNlIE5vbmUsCiAgICAgICAgInBlcl9jbGFzc19mMSI6IHBlcl9jbGFzcywKICAgICAgICAiY29uZnVzaW9uX21hdHJpeCI6IGNvbmZ1c2lvbiwKICAgIH0KCgpkZWYgZXhhY3RfbWNuZW1hcihmaXhlZDogaW50LCBicm9rZW46IGludCkgLT4gZmxvYXQgfCBOb25lOgogICAgZGlzY29yZGFudCA9IGZpeGVkICsgYnJva2VuCiAgICBpZiBkaXNjb3JkYW50ID09IDA6CiAgICAgICAgcmV0dXJuIDEuMAogICAgdGFpbCA9IG1pbihmaXhlZCwgYnJva2VuKQogICAgcHJvYmFiaWxpdHkgPSBzdW0obWF0aC5jb21iKGRpc2NvcmRhbnQsIGspIGZvciBrIGluIHJhbmdlKHRhaWwgKyAxKSkgLyAoMiAqKiBkaXNjb3JkYW50KQogICAgcmV0dXJuIG1pbigxLjAsIDIgKiBwcm9iYWJpbGl0eSkKCgpkZWYgcGFpcmVkKHJvd3M6IGxpc3RbZGljdF0sIG1vZGVsOiBzdHIpIC0+IGRpY3Q6CiAgICBmaXhlZCA9IGJyb2tlbiA9IDAKICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICB0cnV0aCA9IHJvdy5nZXQoImxhYmVsIikKICAgICAgICByYXcgPSBwcmVkaWN0aW9uKHJvdywgInJhdyIsIG1vZGVsKQogICAgICAgIGZyb250YWwgPSBwcmVkaWN0aW9uKHJvdywgImZyb250YWwiLCBtb2RlbCkKICAgICAgICByYXdfY29ycmVjdCA9IHJhdyBpcyBub3QgTm9uZSBhbmQgcmF3LmdldCgibGFiZWwiKSA9PSB0cnV0aAogICAgICAgIGZyb250YWxfY29ycmVjdCA9IGZyb250YWwgaXMgbm90IE5vbmUgYW5kIGZyb250YWwuZ2V0KCJsYWJlbCIpID09IHRydXRoCiAgICAgICAgZml4ZWQgKz0gaW50KG5vdCByYXdfY29ycmVjdCBhbmQgZnJvbnRhbF9jb3JyZWN0KQogICAgICAgIGJyb2tlbiArPSBpbnQocmF3X2NvcnJlY3QgYW5kIG5vdCBmcm9udGFsX2NvcnJlY3QpCiAgICByZXR1cm4geyJmaXhlZF93cm9uZ190b19yaWdodCI6IGZpeGVkLCAiYnJva2VuX3JpZ2h0X3RvX3dyb25nIjogYnJva2VuLAogICAgICAgICAgICAiZGlzY29yZGFudCI6IGZpeGVkICsgYnJva2VuLCAibWNuZW1hcl9leGFjdF90d29fc2lkZWRfcCI6IGV4YWN0X21jbmVtYXIoZml4ZWQsIGJyb2tlbil9CgoKZGVmIHBvc2VfYmluKHJvdzogZGljdCwgZGltZW5zaW9uOiBzdHIpIC0+IHN0ciB8IE5vbmU6CiAgICBwb3NlID0gcm93LmdldCgicG9zZSIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShwb3NlLCBsaXN0KSBvciBsZW4ocG9zZSkgPCAyOgogICAgICAgIHJldHVybiBOb25lCiAgICB2YWx1ZSA9IGFicyhmbG9hdChwb3NlWzAgaWYgZGltZW5zaW9uID09ICJ5YXciIGVsc2UgMV0pKQogICAgcmV0dXJuICI8MTUiIGlmIHZhbHVlIDwgMTUgZWxzZSAoIjE1LTMwIiBpZiB2YWx1ZSA8IDMwIGVsc2UgIj49MzAiKQoKCmRlZiBtZXRyaWNfcmVjb3JkKGFuZ2xlOiBpbnQgfCBOb25lLCBncm91cF90eXBlOiBzdHIsIGdyb3VwOiBzdHIsIGJyYW5jaDogc3RyLCBtb2RlbDogc3RyLCByb3dzOiBsaXN0W2RpY3RdKSAtPiBkaWN0OgogICAgcmVzdWx0ID0gbWV0cmljcyhyb3dzLCBicmFuY2gsIG1vZGVsKQogICAgcmV0dXJuIHsiYW5nbGUiOiBhbmdsZSwgImdyb3VwX3R5cGUiOiBncm91cF90eXBlLCAiZ3JvdXAiOiBncm91cCwgImJyYW5jaCI6IGJyYW5jaCwKICAgICAgICAgICAgIm1vZGVsIjogbW9kZWwsICoqcmVzdWx0fQoKCmRlZiBzdW1tYXJpemUocmVzdWx0c19kaXI6IFBhdGgpIC0+IGRpY3Q6CiAgICByb3dzID0gbG9hZF9yb3dzKHJlc3VsdHNfZGlyKQogICAgbWV0cmljX3Jvd3MgPSBbXQogICAgcGFpcmVkX3Jvd3MgPSBbXQogICAgYnlfYW5nbGU6IGRpY3RbaW50LCBsaXN0W2RpY3RdXSA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgYnlfYW5nbGVbaW50KHJvd1siYW5nbGUiXSldLmFwcGVuZChyb3cpCiAgICBmb3IgYW5nbGUsIHN1YnNldCBpbiBzb3J0ZWQoYnlfYW5nbGUuaXRlbXMoKSk6CiAgICAgICAgZm9yIGJyYW5jaCBpbiBCUkFOQ0hFUzoKICAgICAgICAgICAgZm9yIG1vZGVsIGluIE1PREVMUzoKICAgICAgICAgICAgICAgIG1ldHJpY19yb3dzLmFwcGVuZChtZXRyaWNfcmVjb3JkKGFuZ2xlLCAiYW5nbGUiLCBzdHIoYW5nbGUpLCBicmFuY2gsIG1vZGVsLCBzdWJzZXQpKQogICAgICAgIGZvciBtb2RlbCBpbiBNT0RFTFM6CiAgICAgICAgICAgIHBhaXJlZF9yb3dzLmFwcGVuZCh7ImFuZ2xlIjogYW5nbGUsICJtb2RlbCI6IG1vZGVsLCAqKnBhaXJlZChzdWJzZXQsIG1vZGVsKX0pCgogICAgbmF0dXJhbCA9IGJ5X2FuZ2xlLmdldCgwLCBbXSkKICAgIHBvc2VfZ3JvdXBzID0geyJ5YXciOiAoIjwxNSIsICIxNS0zMCIsICI+PTMwIiksICJwaXRjaCI6ICgiPDE1IiwgIjE1LTMwIiwgIj49MzAiKX0KICAgIGZvciBkaW1lbnNpb24sIGJpbnMgaW4gcG9zZV9ncm91cHMuaXRlbXMoKToKICAgICAgICBmb3IgYmluX25hbWUgaW4gYmluczoKICAgICAgICAgICAgc3Vic2V0ID0gW3JvdyBmb3Igcm93IGluIG5hdHVyYWwgaWYgcG9zZV9iaW4ocm93LCBkaW1lbnNpb24pID09IGJpbl9uYW1lXQogICAgICAgICAgICBpZiBub3Qgc3Vic2V0OgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZm9yIGJyYW5jaCBpbiBCUkFOQ0hFUzoKICAgICAgICAgICAgICAgIGZvciBtb2RlbCBpbiBNT0RFTFM6CiAgICAgICAgICAgICAgICAgICAgbWV0cmljX3Jvd3MuYXBwZW5kKG1ldHJpY19yZWNvcmQoMCwgZGltZW5zaW9uLCBiaW5fbmFtZSwgYnJhbmNoLCBtb2RlbCwgc3Vic2V0KSkKCiAgICBzdGFnZV9jb3VudHM6IENvdW50ZXJbc3RyXSA9IENvdW50ZXIoKQogICAgZXJyb3JfY291bnRzOiBDb3VudGVyW3N0cl0gPSBDb3VudGVyKCkKICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICBmb3Iga2V5IGluICgicmF3X2Vycm9yIiwgImZyb250YWxfZXJyb3IiLCAiZnJvbnRhbF9hbmFseXplX2Vycm9yIiwgImlucHV0X2Vycm9yIik6CiAgICAgICAgICAgIGlmIHJvdy5nZXQoa2V5KToKICAgICAgICAgICAgICAgIHN0YWdlX2NvdW50c1trZXkucmVtb3Zlc3VmZml4KCJfZXJyb3IiKSArICI6ZXJyb3IiXSArPSAxCiAgICAgICAgICAgICAgICBlcnJvcl9jb3VudHNbZiJ7a2V5LnJlbW92ZXN1ZmZpeCgnX2Vycm9yJyl9fHtyb3dba2V5XX0iXSArPSAxCiAgICAgICAgZm9yIGJyYW5jaCBpbiBCUkFOQ0hFUzoKICAgICAgICAgICAgZm9yIG1vZGVsIGluIE1PREVMUzoKICAgICAgICAgICAgICAgIHByZWQgPSAocm93LmdldChicmFuY2gpIG9yIHt9KS5nZXQobW9kZWwpIG9yIHt9CiAgICAgICAgICAgICAgICBpZiAiZXJyb3IiIGluIHByZWQ6CiAgICAgICAgICAgICAgICAgICAgc3RhZ2VfY291bnRzW2Yie2JyYW5jaH06e21vZGVsfTplcnJvciJdICs9IDEKICAgICAgICAgICAgICAgICAgICBlcnJvcl9jb3VudHNbZiJ7YnJhbmNofTp7bW9kZWx9fHtwcmVkWydlcnJvciddfSJdICs9IDEKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgc3RhZ2VfY291bnRzW2Yie2JyYW5jaH06e21vZGVsfTpvayJdICs9IDEKICAgICAgICB0aW1pbmdzID0gcm93LmdldCgidGltaW5ncyIpIG9yIHt9CiAgICAgICAgZm9yIHN0YWdlIGluICgibXNfcmF3IiwgIm1zX2Zyb250YWxpemUiLCAibXNfZnJvbnRhbCIpOgogICAgICAgICAgICBzdGFnZV9jb3VudHNbZiJ7c3RhZ2V9Onsnb2snIGlmIHN0YWdlIGluIHRpbWluZ3MgZWxzZSAnbWlzc2luZyd9Il0gKz0gMQoKICAgIHN1bW1hcnkgPSB7CiAgICAgICAgImRhdGFzZXQiOiAiUkFGLURCIGFsaWduZWQgdGVzdCBzcGxpdCIsCiAgICAgICAgIm5fcm93cyI6IGxlbihyb3dzKSwKICAgICAgICAiYW5nbGVzIjogc29ydGVkKGJ5X2FuZ2xlKSwKICAgICAgICAibWV0cmljX3Jvd3MiOiBtZXRyaWNfcm93cywKICAgICAgICAicGFpcmVkX2NvbXBhcmlzb25zIjogcGFpcmVkX3Jvd3MsCiAgICAgICAgInN0YWdlX2NvdW50cyI6IGRpY3Qoc29ydGVkKHN0YWdlX2NvdW50cy5pdGVtcygpKSksCiAgICAgICAgImVycm9yX2NvdW50cyI6IGRpY3Qoc29ydGVkKGVycm9yX2NvdW50cy5pdGVtcygpKSksCiAgICAgICAgIm5vdGUiOiAiRmFpbGVkIHByZWRpY3Rpb25zIGNvdW50IGFzIGluY29ycmVjdCBmb3IgYWNjdXJhY3koYWxsKSwgbWFjcm8gRjEsIGFuZCBwYWlyZWQgZmxpcHM7IGNvdmVyYWdlIHVzZXMgdmFsaWQgcHJlZGljdGlvbnMuIiwKICAgIH0KICAgIChyZXN1bHRzX2RpciAvICJzdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3VtbWFyeSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MikgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiKQogICAgd3JpdGVfbWV0cmljc19jc3YocmVzdWx0c19kaXIgLyAibWV0cmljcy5jc3YiLCBtZXRyaWNfcm93cykKICAgIHdyaXRlX3ByZWRpY3Rpb25zX2NzdihyZXN1bHRzX2RpciAvICJwcmVkaWN0aW9ucy5jc3YiLCByb3dzKQogICAgcHJpbnRfbWFya2Rvd24obWV0cmljX3Jvd3MsIHBhaXJlZF9yb3dzLCBzdGFnZV9jb3VudHMsIGVycm9yX2NvdW50cykKICAgIHJldHVybiBzdW1tYXJ5CgoKZGVmIHdyaXRlX21ldHJpY3NfY3N2KHBhdGg6IFBhdGgsIHJvd3M6IGxpc3RbZGljdF0pIC0+IE5vbmU6CiAgICBjb2x1bW5zID0gWyJhbmdsZSIsICJncm91cF90eXBlIiwgImdyb3VwIiwgImJyYW5jaCIsICJtb2RlbCIsICJuIiwgIm5fdmFsaWQiLCAibl9mYWlsZWQiLCAiY292ZXJhZ2UiLAogICAgICAgICAgICAgICAiYWNjdXJhY3lfdmFsaWQiLCAiYWNjdXJhY3lfYWxsIiwgIm1hY3JvX2YxIiwgIm1lYW5fcF90cnVlX2xhYmVsIl0gKyBbZiJmMV97bmFtZX0iIGZvciBuYW1lIGluIENMQVNTRVNdCiAgICB3aXRoIHBhdGgub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOC1zaWciLCBuZXdsaW5lPSIiKSBhcyBzdHJlYW06CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoc3RyZWFtLCBmaWVsZG5hbWVzPWNvbHVtbnMpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgICAgIGZsYXQgPSB7a2V5OiByb3cuZ2V0KGtleSkgZm9yIGtleSBpbiBjb2x1bW5zIGlmIG5vdCBrZXkuc3RhcnRzd2l0aCgiZjFfIil9CiAgICAgICAgICAgIGZsYXQudXBkYXRlKHtmImYxX3tuYW1lfSI6IHJvd1sicGVyX2NsYXNzX2YxIl1bbmFtZV1bImYxIl0gZm9yIG5hbWUgaW4gQ0xBU1NFU30pCiAgICAgICAgICAgIHdyaXRlci53cml0ZXJvdyhmbGF0KQoKCmRlZiB3cml0ZV9wcmVkaWN0aW9uc19jc3YocGF0aDogUGF0aCwgcm93czogbGlzdFtkaWN0XSkgLT4gTm9uZToKICAgIGNvbHVtbnMgPSBbImltYWdlIiwgImxhYmVsIiwgImFuZ2xlIiwgInlhdyIsICJwaXRjaCIsICJyb2xsIiwgIm9jY2x1ZGVkIiwgIm1zX3JhdyIsICJtc19mcm9udGFsaXplIiwgIm1zX2Zyb250YWwiLCAiZnJvbnRhbF9lcnJvciIsICJmcm9udGFsX2FuYWx5emVfZXJyb3IiLCAicmF3X2Vycm9yIiwgImlucHV0X2Vycm9yIl0KICAgIGZvciBicmFuY2ggaW4gQlJBTkNIRVM6CiAgICAgICAgZm9yIG1vZGVsIGluIE1PREVMUzoKICAgICAgICAgICAgY29sdW1ucy5leHRlbmQoW2Yie2JyYW5jaH1fe21vZGVsfV9sYWJlbCIsIGYie2JyYW5jaH1fe21vZGVsfV9wcm9icyIsIGYie2JyYW5jaH1fe21vZGVsfV9lcnJvciJdKQogICAgd2l0aCBwYXRoLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgtc2lnIiwgbmV3bGluZT0iIikgYXMgc3RyZWFtOgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1jb2x1bW5zKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgZm9yIHJvdyBpbiByb3dzOgogICAgICAgICAgICBwb3NlID0gcm93LmdldCgicG9zZSIpIG9yIFtOb25lLCBOb25lLCBOb25lXQogICAgICAgICAgICBmbGF0ID0geyJpbWFnZSI6IHJvdy5nZXQoImltYWdlIiksICJsYWJlbCI6IHJvdy5nZXQoImxhYmVsIiksICJhbmdsZSI6IHJvdy5nZXQoImFuZ2xlIiksCiAgICAgICAgICAgICAgICAgICAgInlhdyI6IHBvc2VbMF0sICJwaXRjaCI6IHBvc2VbMV0sICJyb2xsIjogcG9zZVsyXSwgIm9jY2x1ZGVkIjogcm93LmdldCgib2NjbHVkZWQiKSwKICAgICAgICAgICAgICAgICAgICAiZnJvbnRhbF9lcnJvciI6IHJvdy5nZXQoImZyb250YWxfZXJyb3IiLCAiIiksCiAgICAgICAgICAgICAgICAgICAgImZyb250YWxfYW5hbHl6ZV9lcnJvciI6IHJvdy5nZXQoImZyb250YWxfYW5hbHl6ZV9lcnJvciIsICIiKSwKICAgICAgICAgICAgICAgICAgICAicmF3X2Vycm9yIjogcm93LmdldCgicmF3X2Vycm9yIiwgIiIpLCAiaW5wdXRfZXJyb3IiOiByb3cuZ2V0KCJpbnB1dF9lcnJvciIsICIiKX0KICAgICAgICAgICAgZmxhdC51cGRhdGUocm93LmdldCgidGltaW5ncyIpIG9yIHt9KQogICAgICAgICAgICBmb3IgYnJhbmNoIGluIEJSQU5DSEVTOgogICAgICAgICAgICAgICAgZm9yIG1vZGVsIGluIE1PREVMUzoKICAgICAgICAgICAgICAgICAgICBwcmVkID0gKHJvdy5nZXQoYnJhbmNoKSBvciB7fSkuZ2V0KG1vZGVsKSBvciB7fQogICAgICAgICAgICAgICAgICAgIGZsYXRbZiJ7YnJhbmNofV97bW9kZWx9X2xhYmVsIl0gPSBwcmVkLmdldCgibGFiZWwiLCAiIikKICAgICAgICAgICAgICAgICAgICBmbGF0W2Yie2JyYW5jaH1fe21vZGVsfV9wcm9icyJdID0ganNvbi5kdW1wcyhwcmVkLmdldCgicHJvYnMiLCB7fSksIGVuc3VyZV9hc2NpaT1GYWxzZSwgc2VwYXJhdG9ycz0oIiwiLCAiOiIpKQogICAgICAgICAgICAgICAgICAgIGZsYXRbZiJ7YnJhbmNofV97bW9kZWx9X2Vycm9yIl0gPSBwcmVkLmdldCgiZXJyb3IiLCAiIikKICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93KGZsYXQpCgoKZGVmIHByaW50X21hcmtkb3duKG1ldHJpY19yb3dzOiBsaXN0W2RpY3RdLCBwYWlyZWRfcm93czogbGlzdFtkaWN0XSwgc3RhZ2VzOiBDb3VudGVyLCBlcnJvcnM6IENvdW50ZXIpIC0+IE5vbmU6CiAgICBpZiBoYXNhdHRyKHN5cy5zdGRvdXQsICJyZWNvbmZpZ3VyZSIpOgogICAgICAgIHN5cy5zdGRvdXQucmVjb25maWd1cmUoZW5jb2Rpbmc9InV0Zi04IiwgZXJyb3JzPSJyZXBsYWNlIikKICAgIHByaW50KCJcbiMjIyBQZXIgYW5nbGUgw5cgYnJhbmNoIMOXIG1vZGVsIikKICAgIHByaW50KCJ8IGFuZ2xlIHwgYnJhbmNoIHwgbW9kZWwgfCBuIHwgY292ZXJhZ2UgfCBhY2N1cmFjeSh2YWxpZCkgfCBhY2N1cmFjeShhbGwpIHwgbWFjcm8gRjEgfCBtZWFuIHAodHJ1ZSkgfCIpCiAgICBwcmludCgifC0tLTp8LS0tfC0tLXwtLS06fC0tLTp8LS0tOnwtLS06fC0tLTp8LS0tOnwiKQogICAgZm9yIHJvdyBpbiBtZXRyaWNfcm93czoKICAgICAgICBpZiByb3dbImdyb3VwX3R5cGUiXSAhPSAiYW5nbGUiOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHBjdCA9IGxhbWJkYSB2YWx1ZTogIuKAlCIgaWYgdmFsdWUgaXMgTm9uZSBlbHNlIGYiezEwMCAqIHZhbHVlOi4yZn0lIgogICAgICAgIG1lYW5fcCA9ICLigJQiIGlmIHJvd1sibWVhbl9wX3RydWVfbGFiZWwiXSBpcyBOb25lIGVsc2UgZiJ7cm93WydtZWFuX3BfdHJ1ZV9sYWJlbCddOi40Zn0iCiAgICAgICAgcHJpbnQoZiJ8IHtyb3dbJ2dyb3VwJ119IHwge3Jvd1snYnJhbmNoJ119IHwge3Jvd1snbW9kZWwnXX0gfCB7cm93WyduJ119IHwge3BjdChyb3dbJ2NvdmVyYWdlJ10pfSB8IHtwY3Qocm93WydhY2N1cmFjeV92YWxpZCddKX0gfCB7cGN0KHJvd1snYWNjdXJhY3lfYWxsJ10pfSB8IHtwY3Qocm93WydtYWNyb19mMSddKX0gfCB7bWVhbl9wfSB8IikKICAgIHByaW50KCJcbiMjIyBQZXItY2xhc3MgRjEgKCUpIikKICAgIHByaW50KCJ8IGdyb3VwIHR5cGUgfCBncm91cCB8IGFuZ2xlIHwgYnJhbmNoIHwgbW9kZWwgfCAiICsgIiB8ICIuam9pbihDTEFTU0VTKSArICIgfCIpCiAgICBwcmludCgifC0tLXwtLS18LS0tOnwtLS18LS0tfCIgKyAiLS0tOnwiICogbGVuKENMQVNTRVMpKQogICAgZm9yIHJvdyBpbiBtZXRyaWNfcm93czoKICAgICAgICBmMXMgPSAiIHwgIi5qb2luKGYiezEwMCAqIHJvd1sncGVyX2NsYXNzX2YxJ11bbmFtZV1bJ2YxJ106LjJmfSIgZm9yIG5hbWUgaW4gQ0xBU1NFUykKICAgICAgICBwcmludChmInwge3Jvd1snZ3JvdXBfdHlwZSddfSB8IHtyb3dbJ2dyb3VwJ119IHwge3Jvd1snYW5nbGUnXSBpZiByb3dbJ2FuZ2xlJ10gaXMgbm90IE5vbmUgZWxzZSAnJ30gfCB7cm93WydicmFuY2gnXX0gfCB7cm93Wydtb2RlbCddfSB8IHtmMXN9IHwiKQogICAgcHJpbnQoIlxuIyMjIEFuZ2xlIDAgcG9zZSBiaW5zIChyYXcgdnMgZnJvbnRhbCkiKQogICAgcHJpbnQoInwgZGltZW5zaW9uIHwgfHBvc2V8IGJpbiB8IG1vZGVsIHwgbiB8IGJyYW5jaCB8IGNvdmVyYWdlIHwgYWNjdXJhY3kodmFsaWQpIHwgYWNjdXJhY3koYWxsKSB8IG1hY3JvIEYxIHwgbWVhbiBwKHRydWUpIHwiKQogICAgcHJpbnQoInwtLS18LS0tfC0tLXwtLS06fC0tLXwtLS06fC0tLTp8LS0tOnwtLS06fC0tLTp8IikKICAgIGZvciByb3cgaW4gbWV0cmljX3Jvd3M6CiAgICAgICAgaWYgcm93WyJncm91cF90eXBlIl0gPT0gImFuZ2xlIjoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwY3QgPSBsYW1iZGEgdmFsdWU6ICLigJQiIGlmIHZhbHVlIGlzIE5vbmUgZWxzZSBmInsxMDAgKiB2YWx1ZTouMmZ9JSIKICAgICAgICBtZWFuX3AgPSAi4oCUIiBpZiByb3dbIm1lYW5fcF90cnVlX2xhYmVsIl0gaXMgTm9uZSBlbHNlIGYie3Jvd1snbWVhbl9wX3RydWVfbGFiZWwnXTouNGZ9IgogICAgICAgIHByaW50KGYifCB7cm93Wydncm91cF90eXBlJ119IHwge3Jvd1snZ3JvdXAnXX0gfCB7cm93Wydtb2RlbCddfSB8IHtyb3dbJ24nXX0gfCB7cm93WydicmFuY2gnXX0gfCB7cGN0KHJvd1snY292ZXJhZ2UnXSl9IHwge3BjdChyb3dbJ2FjY3VyYWN5X3ZhbGlkJ10pfSB8IHtwY3Qocm93WydhY2N1cmFjeV9hbGwnXSl9IHwge3BjdChyb3dbJ21hY3JvX2YxJ10pfSB8IHttZWFuX3B9IHwiKQogICAgcHJpbnQoIlxuIyMjIFBhaXJlZCByYXcg4oaSIGZyb250YWwiKQogICAgcHJpbnQoInwgYW5nbGUgfCBtb2RlbCB8IGZpeGVkICh3cm9uZ+KGknJpZ2h0KSB8IGJyb2tlbiAocmlnaHTihpJ3cm9uZykgfCBleGFjdCBNY05lbWFyIHAgfCIpCiAgICBwcmludCgifC0tLTp8LS0tfC0tLTp8LS0tOnwtLS06fCIpCiAgICBmb3Igcm93IGluIHBhaXJlZF9yb3dzOgogICAgICAgIHByaW50KGYifCB7cm93WydhbmdsZSddfSB8IHtyb3dbJ21vZGVsJ119IHwge3Jvd1snZml4ZWRfd3JvbmdfdG9fcmlnaHQnXX0gfCB7cm93Wydicm9rZW5fcmlnaHRfdG9fd3JvbmcnXX0gfCB7cm93WydtY25lbWFyX2V4YWN0X3R3b19zaWRlZF9wJ106LjZnfSB8IikKICAgIHByaW50KCJcbiMjIyBTdGFnZS9lcnJvciBjb3VudHMiKQogICAgcHJpbnQoInwgY291bnQgfCBzdGFnZS9zdGF0dXMgb3IgZXJyb3IgfCIpCiAgICBwcmludCgifC0tLTp8LS0tfCIpCiAgICBmb3IgbmFtZSwgY291bnQgaW4gc29ydGVkKHN0YWdlcy5pdGVtcygpKToKICAgICAgICBwcmludChmInwge2NvdW50fSB8IHtuYW1lfSB8IikKICAgIGZvciBuYW1lLCBjb3VudCBpbiBzb3J0ZWQoZXJyb3JzLml0ZW1zKCkpOgogICAgICAgIHByaW50KGYifCB7Y291bnR9IHwge25hbWV9IHwiKQoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IlN1bW1hcml6ZSB0aGUgUkFGLURCIDNEREZBIGZyb250YWxpemF0aW9uIGJlbmNobWFyay4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yZXN1bHRzLWRpciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBzdW1tYXJpemUoYXJncy5yZXN1bHRzX2Rpci5yZXNvbHZlKCkpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK'))
(FRONTALIZE_SCRIPTS / 'launch_frontalize_workers.py').write_bytes(base64.b64decode('ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucw0KDQppbXBvcnQgYXJncGFyc2UNCmltcG9ydCBqc29uDQppbXBvcnQgb3MNCmltcG9ydCBzaWduYWwNCmltcG9ydCBzdWJwcm9jZXNzDQppbXBvcnQgc3lzDQppbXBvcnQgdGltZQ0KaW1wb3J0IHVybGxpYi5lcnJvcg0KaW1wb3J0IHVybGxpYi5yZXF1ZXN0DQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgNCg0KDQpNT0RFTF9OQU1FID0gIkVNT0NBX3YyX2xyX21zZV8yMCINCk1BWF9TVEFSVF9TRUNPTkRTID0gOTAwDQoNCg0KZGVmIGNoZWNrX2hlYWx0aCh1cmw6IHN0cikgLT4gYm9vbDoNCiAgICB0cnk6DQogICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3Blbih1cmwgKyAiL3YxL2hlYWx0aCIsIHRpbWVvdXQ9NSkgYXMgcmVzcG9uc2U6DQogICAgICAgICAgICByZXR1cm4gcmVzcG9uc2Uuc3RhdHVzID09IDIwMA0KICAgIGV4Y2VwdCAodXJsbGliLmVycm9yLlVSTEVycm9yLCBUaW1lb3V0RXJyb3IsIE9TRXJyb3IpOg0KICAgICAgICByZXR1cm4gRmFsc2UNCg0KDQpkZWYgc3RvcF9wcm9jZXNzKHByb2M6IHN1YnByb2Nlc3MuUG9wZW4gfCBOb25lKSAtPiBOb25lOg0KICAgIGlmIHByb2MgaXMgTm9uZSBvciBwcm9jLnBvbGwoKSBpcyBub3QgTm9uZToNCiAgICAgICAgcmV0dXJuDQogICAgdHJ5Og0KICAgICAgICBvcy5raWxscGcocHJvYy5waWQsIHNpZ25hbC5TSUdURVJNKQ0KICAgICAgICBwcm9jLndhaXQodGltZW91dD0yMCkNCiAgICBleGNlcHQgKFByb2Nlc3NMb29rdXBFcnJvciwgc3VicHJvY2Vzcy5UaW1lb3V0RXhwaXJlZCk6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIG9zLmtpbGxwZyhwcm9jLnBpZCwgc2lnbmFsLlNJR0tJTEwpDQogICAgICAgIGV4Y2VwdCBQcm9jZXNzTG9va3VwRXJyb3I6DQogICAgICAgICAgICBwYXNzDQogICAgICAgIHByb2Mud2FpdCh0aW1lb3V0PTEwKQ0KDQoNCmRlZiBtYWluKCkgLT4gaW50Og0KICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPSJSdW4gZnJvbnRhbGl6YXRpb24gYmVuY2htYXJrIHNoYXJkcyB3aXRoIG9uZSBFTU9DQSBzZXJ2ZXIgcGVyIEdQVS4iKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYnVuZGxlLXJvb3QiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS13b3JrZGlyIiwgdHlwZT1QYXRoLCBkZWZhdWx0PVBhdGgoIi9rYWdnbGUvd29ya2luZy9yYWZkYjUiKSkNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1kaXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1saW1pdCIsIHR5cGU9aW50LCBkZWZhdWx0PTApDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ncHVzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MikNCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWFuZ2xlcyIsIGRlZmF1bHQ9IjAsNDUsOTAiKQ0KICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9kZSIsIGNob2ljZXM9KCJmcm9udGFsM2QiLCAicm9sbDJkIiksIGRlZmF1bHQ9ImZyb250YWwzZCIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1taW4tcm9sbCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9NS4wKQ0KICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpDQogICAgaWYgYXJncy5ncHVzIDwgMToNCiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiLS1ncHVzIG11c3QgYmUgYXQgbGVhc3QgMSIpDQogICAgYnVuZGxlLCB3b3JrZGlyID0gYXJncy5idW5kbGVfcm9vdC5yZXNvbHZlKCksIGFyZ3Mud29ya2Rpci5yZXNvbHZlKCkNCiAgICBwYXRoc19maWxlID0gd29ya2RpciAvICJydW50aW1lX3BhdGhzLmpzb24iDQogICAgaWYgbm90IHBhdGhzX2ZpbGUuaXNfZmlsZSgpOg0KICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYiUnVuIHByZXBhcmVfcnVudGltZS5weSBmaXJzdDsgbWlzc2luZyB7cGF0aHNfZmlsZX0iKQ0KICAgIHBhdGhzID0ganNvbi5sb2FkcyhwYXRoc19maWxlLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkNCiAgICBwcm9qZWN0LCBlbW9jYV9zb3VyY2UgPSBQYXRoKHBhdGhzWyJwcm9qZWN0X3Jvb3QiXSksIFBhdGgocGF0aHNbImVtb2NhX3NvdXJjZV9yb290Il0pDQogICAgbW9kZWxfZGlyID0gZW1vY2Ffc291cmNlIC8gImFzc2V0cyIgLyAiRU1PQ0EiIC8gIm1vZGVscyINCiAgICBlbnZfZmlsZSA9IGJ1bmRsZSAvICJlbnZpcm9ubWVudCIgLyAiZW1vY2Etd29yazM4LnltbCINCiAgICBpZiBub3QgKGVtb2NhX3NvdXJjZSAvICIuZ2l0IikuaXNfZGlyKCk6DQogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkVNT0NBIHNvdXJjZSBtaXNzaW5nOyBydW4gc2V0dXBfZW1vY2Ffc291cmNlLnNoIikNCiAgICBpZiBub3QgKG1vZGVsX2RpciAvIE1PREVMX05BTUUpLmlzX2RpcigpOg0KICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYiRU1PQ0EgYXNzZXRzIG1pc3Npbmc7IHJ1biB0aGUgbGljZW5zZS1nYXRlZCBhc3NldCBkb3dubG9hZDoge21vZGVsX2RpciAvIE1PREVMX05BTUV9IikNCiAgICBpZiBub3QgKHByb2plY3QgLyAibW9kZWwiIC8gIm1hM2QiIC8gImNoZWNrcG9pbnRzIiAvICJyYWZfTUEzRC5wdGgiKS5pc19maWxlKCk6DQogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIk1BM0QgY2hlY2twb2ludCBsaW5rIG1pc3Npbmc7IHJlcnVuIHByZXBhcmVfcnVudGltZS5weSIpDQogICAgaWYgbm90IGVudl9maWxlLmlzX2ZpbGUoKToNCiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmIk1pc3NpbmcgRU1PQ0EgZW52aXJvbm1lbnQgZmlsZToge2Vudl9maWxlfSIpDQoNCiAgICBvdXRwdXRfZGlyID0gYXJncy5vdXRwdXRfZGlyLnJlc29sdmUoKQ0KICAgIG91dHB1dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIGNvbmRhID0gb3MuZW52aXJvbi5nZXQoIkNPTkRBX0VYRSIpIG9yICJjb25kYSINCiAgICB3b3JrZXJfc2NyaXB0ID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudCAvICJydW5fZnJvbnRhbGl6ZV93b3JrZXIucHkiDQogICAgc2VydmljZV9sb2dzID0gW10NCiAgICBzZXJ2aWNlczogbGlzdFtzdWJwcm9jZXNzLlBvcGVuXSA9IFtdDQogICAgd29ya2VyczogbGlzdFtzdWJwcm9jZXNzLlBvcGVuXSA9IFtdDQogICAgdHJ5Og0KICAgICAgICBmb3IgZ3B1IGluIHJhbmdlKGFyZ3MuZ3B1cyk6DQogICAgICAgICAgICBwb3J0ID0gODc2NSArIGdwdQ0KICAgICAgICAgICAgc2VydmljZV9lbnYgPSBvcy5lbnZpcm9uLmNvcHkoKQ0KICAgICAgICAgICAgc2VydmljZV9lbnYudXBkYXRlKHsiQ1VEQV9WSVNJQkxFX0RFVklDRVMiOiBzdHIoZ3B1KSwgIlBZVEhPTlBBVEgiOiBzdHIoZW1vY2Ffc291cmNlKSwgIlBZVEhPTlVOQlVGRkVSRUQiOiAiMSJ9KQ0KICAgICAgICAgICAgc2VydmljZV9lbnYuc2V0ZGVmYXVsdCgiRU1PQ0FfV0FSTVVQIiwgIjEiKQ0KICAgICAgICAgICAgbG9nID0gKG91dHB1dF9kaXIgLyBmImVtb2NhX2dwdXtncHV9LmxvZyIpLm9wZW4oImEiLCBlbmNvZGluZz0idXRmLTgiKQ0KICAgICAgICAgICAgc2VydmljZV9sb2dzLmFwcGVuZChsb2cpDQogICAgICAgICAgICBjb21tYW5kID0gW2NvbmRhLCAicnVuIiwgIi0tbm8tY2FwdHVyZS1vdXRwdXQiLCAiLW4iLCAid29yazM4IiwgInB5dGhvbiIsICItdSIsDQogICAgICAgICAgICAgICAgICAgICAgIHN0cihwcm9qZWN0IC8gInNlcnZpY2VzIiAvICJlbW9jYV9zdmMiIC8gInNlcnZlci5weSIpLA0KICAgICAgICAgICAgICAgICAgICAgICAiLS1jb25maWciLCBzdHIocHJvamVjdCAvICJjb25maWdzIiAvICJzZXJ2aWNlcy55YW1sIiksDQogICAgICAgICAgICAgICAgICAgICAgICItLW1vZGVscy1kaXIiLCBzdHIobW9kZWxfZGlyKSwgIi0tbW9kZWwtbmFtZSIsIE1PREVMX05BTUUsICItLXBvcnQiLCBzdHIocG9ydCldDQogICAgICAgICAgICBzZXJ2aWNlcy5hcHBlbmQoc3VicHJvY2Vzcy5Qb3Blbihjb21tYW5kLCBjd2Q9cHJvamVjdCwgZW52PXNlcnZpY2VfZW52LCBzdGRvdXQ9bG9nLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RkZXJyPXN1YnByb2Nlc3MuU1RET1VULCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKSkNCg0KICAgICAgICBmb3IgZ3B1IGluIHJhbmdlKGFyZ3MuZ3B1cyk6DQogICAgICAgICAgICB1cmwgPSBmImh0dHA6Ly8xMjcuMC4wLjE6ezg3NjUgKyBncHV9Ig0KICAgICAgICAgICAgZGVhZGxpbmUgPSB0aW1lLm1vbm90b25pYygpICsgTUFYX1NUQVJUX1NFQ09ORFMNCiAgICAgICAgICAgIHdoaWxlIHRpbWUubW9ub3RvbmljKCkgPCBkZWFkbGluZSBhbmQgbm90IGNoZWNrX2hlYWx0aCh1cmwpOg0KICAgICAgICAgICAgICAgIGlmIHNlcnZpY2VzW2dwdV0ucG9sbCgpIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgICAgICAgICBsb2dfcGF0aCA9IG91dHB1dF9kaXIgLyBmImVtb2NhX2dwdXtncHV9LmxvZyINCiAgICAgICAgICAgICAgICAgICAgdGFpbCA9IGxvZ19wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiLCBlcnJvcnM9InJlcGxhY2UiKVstNTAwMDpdDQogICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkVNT0NBIEdQVSB7Z3B1fSBleGl0ZWQgKHtzZXJ2aWNlc1tncHVdLnJldHVybmNvZGV9KTpcbnt0YWlsfSIpDQogICAgICAgICAgICAgICAgdGltZS5zbGVlcCgzKQ0KICAgICAgICAgICAgaWYgbm90IGNoZWNrX2hlYWx0aCh1cmwpOg0KICAgICAgICAgICAgICAgIHJhaXNlIFRpbWVvdXRFcnJvcihmIkVNT0NBIEdQVSB7Z3B1fSBub3QgaGVhbHRoeSBpbiB7TUFYX1NUQVJUX1NFQ09ORFN9cyIpDQogICAgICAgICAgICBwcmludChmIkVNT0NBIEdQVSB7Z3B1fSBoZWFsdGh5IGF0IHt1cmx9IiwgZmx1c2g9VHJ1ZSkNCg0KICAgICAgICBmb3IgZ3B1IGluIHJhbmdlKGFyZ3MuZ3B1cyk6DQogICAgICAgICAgICB3b3JrZXJfZW52ID0gb3MuZW52aXJvbi5jb3B5KCkNCiAgICAgICAgICAgIHdvcmtlcl9lbnYucG9wKCJQWVRIT05QQVRIIiwgTm9uZSkNCiAgICAgICAgICAgIHdvcmtlcl9lbnZbIkNVREFfVklTSUJMRV9ERVZJQ0VTIl0gPSBzdHIoZ3B1KQ0KICAgICAgICAgICAgd29ya2VyX2VudlsiTUEzRF9ERVZJQ0UiXSA9ICJjdWRhOjAiDQogICAgICAgICAgICB3b3JrZXJfZW52WyJFTU9DQV9VUkwiXSA9IGYiaHR0cDovLzEyNy4wLjAuMTp7ODc2NSArIGdwdX0iDQogICAgICAgICAgICBjb21tYW5kID0gW3N5cy5leGVjdXRhYmxlLCBzdHIod29ya2VyX3NjcmlwdCksICItLWJ1bmRsZS1yb290Iiwgc3RyKGJ1bmRsZSksICItLXByb2plY3Qtcm9vdCIsIHN0cihwcm9qZWN0KSwNCiAgICAgICAgICAgICAgICAgICAgICAgIi0tZW1vY2EtdXJsIiwgd29ya2VyX2VudlsiRU1PQ0FfVVJMIl0sICItLW91dC1qc29ubCIsIHN0cihvdXRwdXRfZGlyIC8gZiJzaGFyZF97Z3B1fS5qc29ubCIpLA0KICAgICAgICAgICAgICAgICAgICAgICAiLS1zaGFyZC1pbmRleCIsIHN0cihncHUpLCAiLS1zaGFyZHMiLCBzdHIoYXJncy5ncHVzKSwgIi0tbGltaXQiLCBzdHIoYXJncy5saW1pdCksDQogICAgICAgICAgICAgICAgICAgICAgICItLWFuZ2xlcyIsIGFyZ3MuYW5nbGVzLCAiLS1tb2RlIiwgYXJncy5tb2RlLCAiLS1taW4tcm9sbCIsIHN0cihhcmdzLm1pbl9yb2xsKV0NCiAgICAgICAgICAgIHdvcmtlcnMuYXBwZW5kKHN1YnByb2Nlc3MuUG9wZW4oY29tbWFuZCwgY3dkPXByb2plY3QsIGVudj13b3JrZXJfZW52KSkNCiAgICAgICAgcHJpbnQoZiJTdGFydGVkIHthcmdzLmdwdXN9IGZyb250YWxpemF0aW9uIHdvcmtlcihzKS4gT3V0cHV0IGRpcmVjdG9yeToge291dHB1dF9kaXJ9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgd2hpbGUgYW55KHdvcmtlci5wb2xsKCkgaXMgTm9uZSBmb3Igd29ya2VyIGluIHdvcmtlcnMpOg0KICAgICAgICAgICAgdGltZS5zbGVlcCg1KQ0KICAgICAgICBzdGF0dXNlcyA9IFt3b3JrZXIucmV0dXJuY29kZSBmb3Igd29ya2VyIGluIHdvcmtlcnNdDQogICAgICAgIHByaW50KGYiV29ya2VyIGV4aXQgY29kZXM6IHtzdGF0dXNlc30iLCBmbHVzaD1UcnVlKQ0KICAgICAgICByZXR1cm4gMCBpZiBhbGwoY29kZSA9PSAwIGZvciBjb2RlIGluIHN0YXR1c2VzKSBlbHNlIDENCiAgICBleGNlcHQgS2V5Ym9hcmRJbnRlcnJ1cHQ6DQogICAgICAgIHByaW50KCJJbnRlcnJ1cHQgcmVjZWl2ZWQ7IEpTT05MIHJvd3MgYXJlIGZsdXNoZWQuIFN0b3BwaW5nIGNoaWxkcmVuLiIsIGZsdXNoPVRydWUpDQogICAgICAgIHJldHVybiAxMzANCiAgICBmaW5hbGx5Og0KICAgICAgICBmb3Igd29ya2VyIGluIHdvcmtlcnM6DQogICAgICAgICAgICBpZiB3b3JrZXIucG9sbCgpIGlzIE5vbmU6DQogICAgICAgICAgICAgICAgd29ya2VyLnRlcm1pbmF0ZSgpDQogICAgICAgIGZvciB3b3JrZXIgaW4gd29ya2VyczoNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICB3b3JrZXIud2FpdCh0aW1lb3V0PTIwKQ0KICAgICAgICAgICAgZXhjZXB0IHN1YnByb2Nlc3MuVGltZW91dEV4cGlyZWQ6DQogICAgICAgICAgICAgICAgd29ya2VyLmtpbGwoKQ0KICAgICAgICBmb3Igc2VydmljZSBpbiBzZXJ2aWNlczoNCiAgICAgICAgICAgIHN0b3BfcHJvY2VzcyhzZXJ2aWNlKQ0KICAgICAgICBmb3IgbG9nIGluIHNlcnZpY2VfbG9nczoNCiAgICAgICAgICAgIGxvZy5jbG9zZSgpDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkNCg=='))

UPSTREAM = PROJECT / 'third_party' / '3DDFA_V2'
UPSTREAM.parent.mkdir(parents=True, exist_ok=True)
PINNED_COMMIT = '1b6c67601abffc1e9f248b291708aef0e43b55ae'
if not (UPSTREAM / '.git').is_dir():
    run_quiet(['git', 'clone', 'https://github.com/cleardusk/3DDFA_V2', str(UPSTREAM)], label='3DDFA-V2 clone')
run_quiet(['git', '-C', str(UPSTREAM), 'checkout', PINNED_COMMIT], label='3DDFA-V2 pinned checkout')
MODEL_DIR = PROJECT / 'model' / '3ddfa_v2'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_SHA256 = {'mb1_120x120.pth': 'a45a946c6e9b16f8d3cf2e69376da9560a7cf9afae671bebceb7e437a405ea79', 'bfm_noneck_v3.pkl': '89ac96480eddc331120f2c8401737c55a5fcc97b10f8a64574e63976c3d246ca', 'param_mean_std_62d_120x120.pkl': '090d7150f77cb66c29ddae21e4508fbde59123dcd1dea7facc24a7ed06d1c795', 'tri.pkl': '0562a594d8369f7d1c86306522ff76fd3a2c81cf90ba9443893c1191ad0abbb8'}
UPSTREAM_PATHS = {
    'mb1_120x120.pth': 'weights/mb1_120x120.pth',
    'bfm_noneck_v3.pkl': 'configs/bfm_noneck_v3.pkl',
    'param_mean_std_62d_120x120.pkl': 'configs/param_mean_std_62d_120x120.pkl',
    'tri.pkl': 'configs/tri.pkl',
}
for filename, relative in UPSTREAM_PATHS.items():
    source = UPSTREAM / relative
    destination = MODEL_DIR / filename
    shutil.copy2(source, destination)
    actual = hashlib.sha256(destination.read_bytes()).hexdigest()
    if actual != EXPECTED_SHA256[filename]:
        raise RuntimeError(f'3DDFA-V2 SHA-256 mismatch for {filename}: expected {EXPECTED_SHA256[filename]}, got {actual}')
    print(f'Verified {filename} sha256={actual}')

# Load the model and frontalize the first sorted RAF-DB test image before EMOCA setup.
import csv, sys
sys.path.insert(0, str(PROJECT))
from PIL import Image
from services.frontalization import get_tddfa_v2_frontalizer
with (BUNDLE_ROOT / 'dataset' / 'test_labels.csv').open(encoding='utf-8-sig', newline='') as stream:
    first = sorted(csv.DictReader(stream), key=lambda row: (int(row['label']), row['image']))[0]
sample_path = BUNDLE_ROOT / 'dataset' / 'DATASET' / 'test' / str(int(first['label'])) / Path(first['image']).name
with Image.open(sample_path) as image:
    frontalized = get_tddfa_v2_frontalizer().frontalize(image.convert('RGB'))
print('3DDFA-V2 load check passed; pose yaw/pitch/roll:', frontalized.yaw_deg, frontalized.pitch_deg, frontalized.roll_deg)


In [ ]:
# Embed the roll-sweep summarizer next to the frontalization scripts.
import base64
(FRONTALIZE_SCRIPTS / 'summarize_roll_sweep.py').write_bytes(base64.b64decode('IiIiU3VtbWFyaXplIHRoZSBBRFItMDI4IHJvbGwtb25seSBzd2VlcCBhbmQgY2hvb3NlIHRoZSByb2xsIHRocmVzaG9sZCBwb3N0IGhvYy4KCklucHV0OiBzaGFyZF8qLmpzb25sIHdyaXR0ZW4gYnkgcnVuX2Zyb250YWxpemVfd29ya2VyLnB5IC0tbW9kZSByb2xsMmQuIEZvciBlYWNoIHJvdyB0aGUKYHJhd2AgYnJhbmNoIGlzIHRoZSByb3RhdGVkIGltYWdlIGFuZCBgZnJvbnRhbGAgaXMgdGhhdCBpbWFnZSBhZnRlciAyRCByb2xsIGNvcnJlY3Rpb24KKGlkZW50aWNhbCB0byByYXcgd2hlbiB8cm9sbHwgPCAtLW1pbi1yb2xsKS4gRm9yIGEgdGhyZXNob2xkIFQgdGhlIGRlcGxveWVkIHJ1bGUgaXM6CnVzZSBgZnJvbnRhbGAgd2hlbiB8ZXN0aW1hdGVkIHJvbGx8ID49IFQsIGVsc2UgYHJhd2AuIEZhaWx1cmVzIChub19mYWNlLCAuLi4pIGNvdW50IGFzIHdyb25nLgpXcml0ZXMgc3VtbWFyeS5qc29uIGFuZCB0aHJlc2hvbGRfc3dlZXAuY3N2OyBwcmludHMgTWFya2Rvd24gdGFibGVzLiBOdW1iZXJzIG9ubHkuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQganNvbgppbXBvcnQgc3RhdGlzdGljcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCkNMQVNTRVMgPSAoInN1cnByaXNlIiwgImZlYXIiLCAiZGlzZ3VzdCIsICJoYXBwaW5lc3MiLCAic2FkbmVzcyIsICJhbmdlciIsICJuZXV0cmFsIikKTU9ERUxTID0gKCJwb3N0ZXJfcmFmZGIiLCAibWEzZF9yYWZkYiIpCkRFRkFVTFRfVEhSRVNIT0xEUyA9ICg1LCAxMCwgMTUsIDIwLCAyNSwgMzAsIDM1LCA0MCwgNTAsIDYwKQoKCmRlZiBsb2FkKHJlc3VsdHNfZGlyOiBQYXRoKSAtPiBsaXN0W2RpY3RdOgogICAgbGF0ZXN0ID0ge30KICAgIGZvciBwYXRoIGluIHNvcnRlZChyZXN1bHRzX2Rpci5nbG9iKCJzaGFyZF8qLmpzb25sIikpOgogICAgICAgIGZvciBsaW5lIGluIHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpLnNwbGl0bGluZXMoKToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcm93ID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgICAgICAgICAgbGF0ZXN0Wyhyb3dbImltYWdlIl0sIGludChyb3dbImFuZ2xlIl0pKV0gPSByb3cKICAgICAgICAgICAgZXhjZXB0IChqc29uLkpTT05EZWNvZGVFcnJvciwgS2V5RXJyb3IsIFR5cGVFcnJvciwgVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgaWYgbm90IGxhdGVzdDoKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYiTm8gc2hhcmRfKi5qc29ubCByb3dzIGluIHtyZXN1bHRzX2Rpcn0iKQogICAgcmV0dXJuIGxpc3QobGF0ZXN0LnZhbHVlcygpKQoKCmRlZiBsYWJlbChyb3c6IGRpY3QsIGJyYW5jaDogc3RyLCBtb2RlbDogc3RyKSAtPiBzdHIgfCBOb25lOgogICAgdmFsdWUgPSAocm93LmdldChicmFuY2gpIG9yIHt9KS5nZXQobW9kZWwpIG9yIHt9CiAgICByZXR1cm4gdmFsdWUuZ2V0KCJsYWJlbCIpIGlmIHZhbHVlLmdldCgibGFiZWwiKSBpbiBDTEFTU0VTIGVsc2UgTm9uZQoKCmRlZiByb2xsKHJvdzogZGljdCkgLT4gZmxvYXQgfCBOb25lOgogICAgcG9zZSA9IHJvdy5nZXQoInBvc2UiKQogICAgcmV0dXJuIGZsb2F0KHBvc2VbMl0pIGlmIHBvc2UgZWxzZSBOb25lCgoKZGVmIGNob3Nlbihyb3c6IGRpY3QsIG1vZGVsOiBzdHIsIHRocmVzaG9sZDogZmxvYXQgfCBOb25lKSAtPiBzdHIgfCBOb25lOgogICAgIiIidGhyZXNob2xkIE5vbmUgPSByYXcgb25seTsgMCA9IGFsd2F5cyBjb3JyZWN0ZWQ7IGVsc2UgdGhlIGRlcGxveWVkIHJ1bGUuIiIiCiAgICBpZiB0aHJlc2hvbGQgaXMgTm9uZToKICAgICAgICByZXR1cm4gbGFiZWwocm93LCAicmF3IiwgbW9kZWwpCiAgICByID0gcm9sbChyb3cpCiAgICBpZiByIGlzIG5vdCBOb25lIGFuZCBhYnMocikgPj0gdGhyZXNob2xkOgogICAgICAgIHJldHVybiBsYWJlbChyb3csICJmcm9udGFsIiwgbW9kZWwpCiAgICByZXR1cm4gbGFiZWwocm93LCAicmF3IiwgbW9kZWwpCgoKZGVmIG1hY3JvX2YxKHJvd3M6IGxpc3RbZGljdF0sIG1vZGVsOiBzdHIsIHRocmVzaG9sZDogZmxvYXQgfCBOb25lKSAtPiBmbG9hdDoKICAgIGYxcyA9IFtdCiAgICBmb3IgbmFtZSBpbiBDTEFTU0VTOgogICAgICAgIHRwID0gc3VtKDEgZm9yIHIgaW4gcm93cyBpZiByWyJsYWJlbCJdID09IG5hbWUgYW5kIGNob3NlbihyLCBtb2RlbCwgdGhyZXNob2xkKSA9PSBuYW1lKQogICAgICAgIHByZWRpY3RlZCA9IHN1bSgxIGZvciByIGluIHJvd3MgaWYgY2hvc2VuKHIsIG1vZGVsLCB0aHJlc2hvbGQpID09IG5hbWUpCiAgICAgICAgc3VwcG9ydCA9IHN1bSgxIGZvciByIGluIHJvd3MgaWYgclsibGFiZWwiXSA9PSBuYW1lKQogICAgICAgIHAgPSB0cCAvIHByZWRpY3RlZCBpZiBwcmVkaWN0ZWQgZWxzZSAwLjAKICAgICAgICByYyA9IHRwIC8gc3VwcG9ydCBpZiBzdXBwb3J0IGVsc2UgMC4wCiAgICAgICAgZjFzLmFwcGVuZCgyICogcCAqIHJjIC8gKHAgKyByYykgaWYgcCArIHJjIGVsc2UgMC4wKQogICAgcmV0dXJuIHN1bShmMXMpIC8gbGVuKGYxcykKCgpkZWYgc3RhdHMocm93czogbGlzdFtkaWN0XSwgbW9kZWw6IHN0ciwgdGhyZXNob2xkOiBmbG9hdCB8IE5vbmUpIC0+IGRpY3Q6CiAgICBuID0gbGVuKHJvd3MpCiAgICBwcmVkcyA9IFtjaG9zZW4ociwgbW9kZWwsIHRocmVzaG9sZCkgZm9yIHIgaW4gcm93c10KICAgIHZhbGlkID0gc3VtKHAgaXMgbm90IE5vbmUgZm9yIHAgaW4gcHJlZHMpCiAgICBjb3JyZWN0ID0gc3VtKHAgPT0gclsibGFiZWwiXSBmb3IgcCwgciBpbiB6aXAocHJlZHMsIHJvd3MpKQogICAgcmV0dXJuIHsibiI6IG4sICJjb3ZlcmFnZSI6IHZhbGlkIC8gbiwgImFjY3VyYWN5X2FsbCI6IGNvcnJlY3QgLyBuLCAibWFjcm9fZjEiOiBtYWNyb19mMShyb3dzLCBtb2RlbCwgdGhyZXNob2xkKX0KCgpkZWYgbWFpbigpIC0+IGludDoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18sIGZvcm1hdHRlcl9jbGFzcz1hcmdwYXJzZS5SYXdEZXNjcmlwdGlvbkhlbHBGb3JtYXR0ZXIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXJlc3VsdHMtZGlyIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10aHJlc2hvbGRzIiwgZGVmYXVsdD0iLCIuam9pbihtYXAoc3RyLCBERUZBVUxUX1RIUkVTSE9MRFMpKSkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICB0aHJlc2hvbGRzID0gW2Zsb2F0KHQpIGZvciB0IGluIGFyZ3MudGhyZXNob2xkcy5zcGxpdCgiLCIpIGlmIHQuc3RyaXAoKV0KICAgIHJvd3MgPSBsb2FkKGFyZ3MucmVzdWx0c19kaXIucmVzb2x2ZSgpKQogICAgYW5nbGVzID0gc29ydGVkKHtpbnQoclsiYW5nbGUiXSkgZm9yIHIgaW4gcm93c30pCiAgICBieV9hbmdsZSA9IHthOiBbciBmb3IgciBpbiByb3dzIGlmIGludChyWyJhbmdsZSJdKSA9PSBhXSBmb3IgYSBpbiBhbmdsZXN9CgogICAgIyBSb2xsIGVzdGltYXRvciBjaGVjazogZXN0aW1hdGVkIHJvbGwgdnMgYXBwbGllZCByb3RhdGlvbiAodXByaWdodCBiYXNlbGluZSBzdWJ0cmFjdGVkIHBlciBpbWFnZSkuCiAgICB1cHJpZ2h0ID0ge3JbImltYWdlIl06IHJvbGwocikgZm9yIHIgaW4gYnlfYW5nbGUuZ2V0KDAsIFtdKSBpZiByb2xsKHIpIGlzIG5vdCBOb25lfQogICAgZXN0aW1hdG9yID0ge30KICAgIGZvciBhIGluIGFuZ2xlczoKICAgICAgICBlcnJzID0gW3JvbGwocikgLSBhIC0gdXByaWdodFtyWyJpbWFnZSJdXSBmb3IgciBpbiBieV9hbmdsZVthXSBpZiByb2xsKHIpIGlzIG5vdCBOb25lIGFuZCByWyJpbWFnZSJdIGluIHVwcmlnaHRdCiAgICAgICAgcm9sbHMgPSBbcm9sbChyKSBmb3IgciBpbiBieV9hbmdsZVthXSBpZiByb2xsKHIpIGlzIG5vdCBOb25lXQogICAgICAgIGVzdGltYXRvclthXSA9IHsibWVkaWFuX3JvbGwiOiBzdGF0aXN0aWNzLm1lZGlhbihyb2xscykgaWYgcm9sbHMgZWxzZSBOb25lLAogICAgICAgICAgICAgICAgICAgICAgICAibWVkaWFuX2Fic19lcnJvcl92c190cnVlIjogc3RhdGlzdGljcy5tZWRpYW4oW2FicyhlKSBmb3IgZSBpbiBlcnJzXSkgaWYgZXJycyBlbHNlIE5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgICJzaGFyZV9hYnNfZXJyb3JfZ3RfMTUiOiAoc3VtKGFicyhlKSA+IDE1IGZvciBlIGluIGVycnMpIC8gbGVuKGVycnMpKSBpZiBlcnJzIGVsc2UgTm9uZX0KCiAgICBwb2xpY2llcyA9IFsoInJhdyIsIE5vbmUpLCAoImFsd2F5cyIsIDAuMCldICsgWyhmIlQ9e3Q6Z30iLCB0KSBmb3IgdCBpbiB0aHJlc2hvbGRzXQogICAgdGFibGUgPSBbXQogICAgZm9yIG1vZGVsIGluIE1PREVMUzoKICAgICAgICBmb3IgbmFtZSwgdCBpbiBwb2xpY2llczoKICAgICAgICAgICAgcGVyX2FuZ2xlID0ge2E6IHN0YXRzKGJ5X2FuZ2xlW2FdLCBtb2RlbCwgdCkgZm9yIGEgaW4gYW5nbGVzfQogICAgICAgICAgICBtZWFuX2FjYyA9IHN1bShzWyJhY2N1cmFjeV9hbGwiXSBmb3IgcyBpbiBwZXJfYW5nbGUudmFsdWVzKCkpIC8gbGVuKHBlcl9hbmdsZSkKICAgICAgICAgICAgZm9yIGEsIHMgaW4gcGVyX2FuZ2xlLml0ZW1zKCk6CiAgICAgICAgICAgICAgICB0YWJsZS5hcHBlbmQoeyJtb2RlbCI6IG1vZGVsLCAicG9saWN5IjogbmFtZSwgInRocmVzaG9sZCI6IHQsICJhbmdsZSI6IGEsICoqc30pCiAgICAgICAgICAgIHRhYmxlLmFwcGVuZCh7Im1vZGVsIjogbW9kZWwsICJwb2xpY3kiOiBuYW1lLCAidGhyZXNob2xkIjogdCwgImFuZ2xlIjogIm1lYW4iLCAibiI6IE5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgImNvdmVyYWdlIjogc3VtKHNbImNvdmVyYWdlIl0gZm9yIHMgaW4gcGVyX2FuZ2xlLnZhbHVlcygpKSAvIGxlbihwZXJfYW5nbGUpLAogICAgICAgICAgICAgICAgICAgICAgICAgICJhY2N1cmFjeV9hbGwiOiBtZWFuX2FjYywKICAgICAgICAgICAgICAgICAgICAgICAgICAibWFjcm9fZjEiOiBzdW0oc1sibWFjcm9fZjEiXSBmb3IgcyBpbiBwZXJfYW5nbGUudmFsdWVzKCkpIC8gbGVuKHBlcl9hbmdsZSl9KQoKICAgIGRlZiBhY2MobW9kZWwsIHBvbGljeSwgYW5nbGUpOgogICAgICAgIHJldHVybiBuZXh0KHJbImFjY3VyYWN5X2FsbCJdIGZvciByIGluIHRhYmxlIGlmIHJbIm1vZGVsIl0gPT0gbW9kZWwgYW5kIHJbInBvbGljeSJdID09IHBvbGljeSBhbmQgclsiYW5nbGUiXSA9PSBhbmdsZSkKCiAgICByZWNvbW1lbmRhdGlvbiA9IHt9CiAgICBmb3IgbW9kZWwgaW4gTU9ERUxTOgogICAgICAgIHJhdzAgPSBhY2MobW9kZWwsICJyYXciLCAwKSBpZiAwIGluIGFuZ2xlcyBlbHNlIE5vbmUKICAgICAgICBjYW5kaWRhdGVzID0gW10KICAgICAgICBmb3IgbmFtZSwgdCBpbiBwb2xpY2llc1syOl06CiAgICAgICAgICAgIGxvc3MwID0gKHJhdzAgLSBhY2MobW9kZWwsIG5hbWUsIDApKSBpZiByYXcwIGlzIG5vdCBOb25lIGVsc2UgMC4wCiAgICAgICAgICAgIGNhbmRpZGF0ZXMuYXBwZW5kKChhY2MobW9kZWwsIG5hbWUsICJtZWFuIiksIC1sb3NzMCwgbmFtZSwgbG9zczApKQogICAgICAgICMgQmVzdCBtZWFuIGFjY3VyYWN5IG92ZXIgYW5nbGVzIGFtb25nIHRocmVzaG9sZHMgdGhhdCBjb3N0IDw9IDAuMiBwb2ludHMgb24gdXByaWdodCBpbWFnZXMuCiAgICAgICAgb2sgPSBbYyBmb3IgYyBpbiBjYW5kaWRhdGVzIGlmIGNbM10gPD0gMC4wMDJdIG9yIGNhbmRpZGF0ZXMKICAgICAgICBiZXN0ID0gbWF4KG9rKQogICAgICAgIHJlY29tbWVuZGF0aW9uW21vZGVsXSA9IHsicG9saWN5IjogYmVzdFsyXSwgIm1lYW5fYWNjdXJhY3lfYWxsIjogYmVzdFswXSwgInVwcmlnaHRfbG9zc19wb2ludHMiOiAxMDAgKiBiZXN0WzNdfQoKICAgIG91dCA9IGFyZ3MucmVzdWx0c19kaXIucmVzb2x2ZSgpCiAgICB3aXRoIChvdXQgLyAidGhyZXNob2xkX3N3ZWVwLmNzdiIpLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSIiKSBhcyBzdHJlYW06CiAgICAgICAgd3JpdGVyID0gY3N2LkRpY3RXcml0ZXIoc3RyZWFtLCBmaWVsZG5hbWVzPVsibW9kZWwiLCAicG9saWN5IiwgInRocmVzaG9sZCIsICJhbmdsZSIsICJuIiwgImNvdmVyYWdlIiwgImFjY3VyYWN5X2FsbCIsICJtYWNyb19mMSJdKQogICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgd3JpdGVyLndyaXRlcm93cyh0YWJsZSkKICAgIHN1bW1hcnkgPSB7Im5fcm93cyI6IGxlbihyb3dzKSwgImFuZ2xlcyI6IGFuZ2xlcywgImVzdGltYXRvciI6IGVzdGltYXRvciwgInRhYmxlIjogdGFibGUsCiAgICAgICAgICAgICAgICJyZWNvbW1lbmRhdGlvbiI6IHJlY29tbWVuZGF0aW9uLAogICAgICAgICAgICAgICAibm90ZSI6ICJGYWlsdXJlcyBjb3VudCBhcyB3cm9uZy4gUnVsZTogY29ycmVjdGVkIGlmIHxlc3RpbWF0ZWQgcm9sbHwgPj0gVCBlbHNlIHJhdy4gIgogICAgICAgICAgICAgICAgICAgICAgICJSZWNvbW1lbmRhdGlvbiA9IGJlc3QgbWVhbiBhY2N1cmFjeSBvdmVyIGFuZ2xlcyB3aXRoIDw9IDAuMiBwdCBsb3NzIGF0IDAgZGVnLiJ9CiAgICAob3V0IC8gInN1bW1hcnkuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhzdW1tYXJ5LCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCgogICAgcHJpbnQoIlxuIyMjIELhu5kgxrDhu5tjIGzGsOG7o25nIHJvbGwgKDNEREZBLVYyKSIpCiAgICBwcmludCgifCBnw7NjIHhvYXkgfCByb2xsIHRydW5nIHbhu4sgfCBzYWkgc+G7kSB0dXnhu4d0IMSR4buRaSB0cnVuZyB24buLIHwgJSBzYWkgPiAxNcKwIHwiKQogICAgcHJpbnQoInwtLS18LS0tfC0tLXwtLS18IikKICAgIGZvciBhLCBlIGluIGVzdGltYXRvci5pdGVtcygpOgogICAgICAgIGYgPSBsYW1iZGEgdiwgcGN0PUZhbHNlOiAiLSIgaWYgdiBpcyBOb25lIGVsc2UgKGYiezEwMCAqIHY6LjFmfSAlIiBpZiBwY3QgZWxzZSBmInt2Oi4xZn3CsCIpCiAgICAgICAgcHJpbnQoZiJ8IHthfcKwIHwge2YoZVsnbWVkaWFuX3JvbGwnXSl9IHwge2YoZVsnbWVkaWFuX2Fic19lcnJvcl92c190cnVlJ10pfSB8IHtmKGVbJ3NoYXJlX2Fic19lcnJvcl9ndF8xNSddLCBUcnVlKX0gfCIpCiAgICBmb3IgbW9kZWwgaW4gTU9ERUxTOgogICAgICAgIHByaW50KGYiXG4jIyMge21vZGVsfTogQWNjKGFsbCkgdGhlbyBjaMOtbmggc8OhY2ggw5cgZ8OzYyIpCiAgICAgICAgcHJpbnQoInwgY2jDrW5oIHPDoWNoIHwgIiArICIgfCAiLmpvaW4oZiJ7YX3CsCIgZm9yIGEgaW4gYW5nbGVzKSArICIgfCB0cnVuZyBiw6xuaCB8IG1hY3JvIEYxIFRCIHwiKQogICAgICAgIHByaW50KCJ8LS0tIiAqIChsZW4oYW5nbGVzKSArIDMpICsgInwiKQogICAgICAgIGZvciBuYW1lLCBfIGluIHBvbGljaWVzOgogICAgICAgICAgICBjZWxscyA9IFtmInsxMDAgKiBhY2MobW9kZWwsIG5hbWUsIGEpOi4xZn0iIGZvciBhIGluIGFuZ2xlc10KICAgICAgICAgICAgbWVhbl9yb3cgPSBuZXh0KHIgZm9yIHIgaW4gdGFibGUgaWYgclsibW9kZWwiXSA9PSBtb2RlbCBhbmQgclsicG9saWN5Il0gPT0gbmFtZSBhbmQgclsiYW5nbGUiXSA9PSAibWVhbiIpCiAgICAgICAgICAgIHByaW50KGYifCB7bmFtZX0gfCAiICsgIiB8ICIuam9pbihjZWxscykgKyBmIiB8IHsxMDAgKiBtZWFuX3Jvd1snYWNjdXJhY3lfYWxsJ106LjFmfSB8IHsxMDAgKiBtZWFuX3Jvd1snbWFjcm9fZjEnXTouMWZ9IHwiKQogICAgICAgIHJlYyA9IHJlY29tbWVuZGF0aW9uW21vZGVsXQogICAgICAgIHByaW50KGYiXG7EkOG7gSB4deG6pXQge21vZGVsfToge3JlY1sncG9saWN5J119IChBY2MgVEIgezEwMCAqIHJlY1snbWVhbl9hY2N1cmFjeV9hbGwnXTouMWZ9ICUsIG3huqV0IOG7nyAwwrA6IHtyZWNbJ3VwcmlnaHRfbG9zc19wb2ludHMnXTouMmZ9IMSRaeG7g20pIikKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo='))
print('Roll-sweep summarizer installed.')


In [ ]:
# Build the project's pinned Python 3.8 / Torch 1.11 / PyTorch3D EMOCA environment.
# Reuse Miniconda if a previous setup attempt already created it under /kaggle/working.
WORKDIR.mkdir(parents=True, exist_ok=True)
# Clear large unused assets from a previous upstream download before setup touches the disk.
asset_root = WORKDIR / 'project' / 'model' / 'emoca' / 'assets'
for relative in (
    'EMOCA/models/EMOCA.zip', 'EMOCA/models/EMOCA_v2_mp.zip',
    'EMOCA/models/EMOCA_v2_lr_cos_1.5.zip', 'data/EMOCA_test_example_data.zip',
    'DECA.zip', 'FaceRecognition.zip', 'FLAME.zip',
    'EMOCA/models/EMOCA', 'EMOCA/models/EMOCA_v2_mp',
    'EMOCA/models/EMOCA_v2_lr_cos_1.5', 'data/EMOCA_test_example_data',
):
    old_asset = asset_root / relative
    if old_asset.is_dir():
        shutil.rmtree(old_asset)
    elif old_asset.exists():
        old_asset.unlink()
try:
    (asset_root / 'data').rmdir()
except OSError:
    pass
miniconda = WORKDIR / 'miniconda'
miniconda_conda = miniconda / 'bin' / 'conda'
if miniconda_conda.is_file():
    os.environ['PATH'] = str(miniconda / 'bin') + os.pathsep + os.environ.get('PATH', '')
    os.environ['CONDA_EXE'] = str(miniconda_conda)
elif shutil.which('conda') is None:
    installer = WORKDIR / 'miniconda.sh'
    subprocess.run(['wget', '-q', 'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh', '-O', str(installer)], check=True)
    subprocess.run(['bash', str(installer), '-b', '-p', str(miniconda)], check=True)
    os.environ['PATH'] = str(miniconda / 'bin') + os.pathsep + os.environ.get('PATH', '')
    os.environ['CONDA_EXE'] = str(miniconda_conda)
else:
    os.environ.setdefault('CONDA_EXE', shutil.which('conda'))
env_setup = BUNDLE_ROOT / 'environment' / 'setup_emoca_env.sh'
env_setup_text = env_setup.read_text(encoding='utf-8')
# Avoid Kaggle's unrelated Python mamba test-runner command and cross-version PYTHONPATH.
env_setup_text = env_setup_text.replace(
    'eval "$(conda shell.bash hook)"',
    'CONDA_EXE="$(command -v conda)"\n'
    'CONDA_BIN="$(dirname "$CONDA_EXE")"\n'
    'export PATH="$CONDA_BIN:$PATH"\n'
    'unset PYTHONPATH\n'
    'eval "$("$CONDA_EXE" shell.bash hook)"',
)
env_setup_text = env_setup_text.replace(
    'if ! command -v mamba >/dev/null; then',
    'if [[ ! -x "$CONDA_BIN/mamba" ]]; then',
)
env_setup_text = env_setup_text.replace(
    '  conda install -n base -c conda-forge mamba=2.3.3 -y',
    '  "$CONDA_EXE" install -n base -c conda-forge mamba=2.3.3 -y',
)
env_setup_text = env_setup_text.replace(
    '  mamba create -n work38 python=3.8 -y',
    '  "$CONDA_BIN/mamba" create -n work38 python=3.8 -y',
)
env_setup_text = env_setup_text.replace(
    'mamba env update -n work38 --file',
    '"$CONDA_BIN/mamba" env update -n work38 --file',
)
# Apply package fixes even if this notebook is attached to an older uploaded Dataset.
setup_lines = []
has_numpy_pin = 'python -m pip install numpy==1.23.5' in env_setup_text
found_pip_upgrade = False
for line in env_setup_text.splitlines():
    if 'python -m pip install --upgrade pip==' in line:
        found_pip_upgrade = True
        setup_lines.append(line)
        if not has_numpy_pin:
            setup_lines.append('conda run -n work38 python -m pip install numpy==1.23.5')
        continue
    if line.rstrip().endswith('python -m pip install Cython==0.29'):
        setup_lines.append(line.replace('Cython==0.29', 'Cython==0.29.37'))
        continue
    if 'python -m pip install onnx==1.10.2 onnxruntime-gpu' in line:
        setup_lines.append(line.replace(
            'onnx==1.10.2 onnxruntime-gpu ', ''
        ).replace('mediapipe ', 'mediapipe==0.10.9 '))
    elif '^(onnx|youtube-dl|mediapipe|mmcv-full|pyzmq)' in line:
        setup_lines.append(line.replace(
            '^(onnx|youtube-dl|mediapipe|mmcv-full|pyzmq)',
            '^(onnx|onnxruntime-gpu|youtube-dl|mediapipe|mmcv-full|pyzmq|opencv-python|opencv-python-headless|opencv-contrib-python|opencv-contrib-python-headless|insightface|deepspeech-gpu)',
        ))
    elif 'python -m pip install mmcv-full -f ' in line:
        continue  # Not imported anywhere in this benchmark's inference path.
    else:
        setup_lines.append(line)
if not found_pip_upgrade and not has_numpy_pin:
    raise RuntimeError('Could not place the pinned NumPy before installing EMOCA requirements.')
env_setup_text = '\n'.join(setup_lines) + '\n'
if 'opencv-python-headless==4.5.5.64' not in env_setup_text:
    editable_install = 'conda run -n work38 python -m pip install -e "$EMOCA_ROOT"'
    if editable_install not in env_setup_text:
        raise RuntimeError('Could not find EMOCA editable-install line for OpenCV normalization.')
    opencv_normalization = (
        editable_install + '\n'
        'conda run -n work38 python -m pip uninstall -y opencv-python opencv-contrib-python opencv-contrib-python-headless opencv-python-headless || true\n'
        'conda run -n work38 python -m pip install opencv-python-headless==4.5.5.64\n'
        'conda run -n work38 python -c "import cv2, mediapipe; print(cv2.__version__, mediapipe.__version__)"\n'
    )
    env_setup_text = env_setup_text.replace(editable_install, opencv_normalization, 1)
if 'tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main' not in env_setup_text:
    conda_hook = 'eval "$("$CONDA_EXE" shell.bash hook)"'
    if conda_hook not in env_setup_text:
        raise RuntimeError('Could not locate Conda hook to add the authorized Anaconda ToS acceptance.')
    tos_acceptance = (
        conda_hook + '\n'
        '# User-authorized acceptance for the Anaconda channels listed by EMOCA.\n'
        '"$CONDA_EXE" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main\n'
        '"$CONDA_EXE" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r'
    )
    env_setup_text = env_setup_text.replace(conda_hook, tos_acceptance, 1)
if 'mamba create -n work38' in env_setup_text:
    raise RuntimeError('Could not patch the bundled Mamba command for Kaggle.')
env_setup_runtime = WORKDIR / 'setup_emoca_env_kaggle.sh'
env_setup_runtime.write_text(env_setup_text, encoding='utf-8')
setup_env = os.environ.copy()
setup_env['CONDA_EXE'] = setup_env.get('CONDA_EXE') or shutil.which('conda') or ''
if not setup_env['CONDA_EXE']:
    raise RuntimeError('Conda executable not found after Miniconda setup.')
setup_env['PATH'] = str(Path(setup_env['CONDA_EXE']).parent) + os.pathsep + setup_env.get('PATH', '')
setup_env.pop('PYTHONPATH', None)
os.environ.pop('PYTHONPATH', None)
run_quiet([setup_env['CONDA_EXE'], 'clean', '--all', '--yes'],
          env=setup_env, label='Conda cache cleanup')
run_quiet(['bash', str(env_setup_runtime), str(BUNDLE_ROOT), str(WORKDIR)],
          env=setup_env, label='EMOCA environment setup')


In [ ]:
# License-gated download: fetch only benchmark assets, not upstream demos/alternate checkpoints.
if not ACCEPT_FLAME_EMOCA_LICENSES:
    raise RuntimeError('Asset download is disabled. Enable it only after personally accepting FLAME and EMOCA terms.')
import base64
asset_env = os.environ.copy()
asset_env['ACCEPT_FLAME_EMOCA_LICENSES'] = '1'
asset_setup = WORKDIR / 'download_emoca_assets_minimal.sh'
asset_setup.write_bytes(base64.b64decode('IyEvdXNyL2Jpbi9lbnYgYmFzaApzZXQgLWV1byBwaXBlZmFpbAoKV09SS0RJUj0iJHsxOj91c2FnZTogZG93bmxvYWRfZW1vY2FfYXNzZXRzLnNoIFdPUktESVJ9IgppZiBbWyAiJHtBQ0NFUFRfRkxBTUVfRU1PQ0FfTElDRU5TRVM6LTF9IiAhPSAiMSIgXV07IHRoZW4KICBjYXQgPiYyIDw8J0VPRicKQmxvY2tlZCBieSBBQ0NFUFRfRkxBTUVfRU1PQ0FfTElDRU5TRVM9MC4gU2V0IGl0IHRvIDEgb25seSBhZnRlciBwZXJzb25hbGx5IGFjY2VwdGluZyB0aGUgRkxBTUUgYW5kIEVNT0NBIHRlcm1zLgpFT0YKICBleGl0IDEwCmZpCgpFTU9DQV9ST09UPSIkV09SS0RJUi9wcm9qZWN0L21vZGVsL2Vtb2NhIgpBU1NFVF9ST09UPSIkRU1PQ0FfUk9PVC9hc3NldHMiCnRlc3QgLWQgIiRFTU9DQV9ST09UL2dkbF9hcHBzL0VNT0NBL2RlbW9zIiB8fCB7IGVjaG8gIlBpbm5lZCBFTU9DQSBzb3VyY2UgaXMgbWlzc2luZy4iID4mMjsgZXhpdCAyOyB9CmNvbW1hbmQgLXYgd2dldCA+L2Rldi9udWxsIHx8IHsgZWNobyAid2dldCBpcyByZXF1aXJlZC4iID4mMjsgZXhpdCAyOyB9CmNvbW1hbmQgLXYgdW56aXAgPi9kZXYvbnVsbCB8fCB7IGVjaG8gInVuemlwIGlzIHJlcXVpcmVkLiIgPiYyOyBleGl0IDI7IH0KCiMgVGhlIGJlbmNobWFyayBvbmx5IGxvYWRzIEVNT0NBX3YyX2xyX21zZV8yMC4gUmVtb3ZlIHVudXNlZCBkZW1vIGRhdGEsCiMgbGVnYWN5L2FsdGVybmF0ZSBjaGVja3BvaW50cyBhbmQgc3RhbGUgYXJjaGl2ZXMgbGVmdCBieSB0aGUgZmFpbGVkIHJ1bi4Kcm0gLWYgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscy9FTU9DQS56aXAiIFwKICAiJEFTU0VUX1JPT1QvRU1PQ0EvbW9kZWxzL0VNT0NBX3YyX21wLnppcCIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfY29zXzEuNS56aXAiIFwKICAiJEFTU0VUX1JPT1QvZGF0YS9FTU9DQV90ZXN0X2V4YW1wbGVfZGF0YS56aXAiIFwKICAiJEFTU0VUX1JPT1QvREVDQS56aXAiICIkQVNTRVRfUk9PVC9GYWNlUmVjb2duaXRpb24uemlwIiAiJEFTU0VUX1JPT1QvRkxBTUUuemlwIgpybSAtcmYgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscy9FTU9DQSIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbXAiIFwKICAiJEFTU0VUX1JPT1QvRU1PQ0EvbW9kZWxzL0VNT0NBX3YyX2xyX2Nvc18xLjUiIFwKICAiJEFTU0VUX1JPT1QvZGF0YS9FTU9DQV90ZXN0X2V4YW1wbGVfZGF0YSIKcm1kaXIgIiRBU1NFVF9ST09UL2RhdGEiIDI+L2Rldi9udWxsIHx8IHRydWUKCiMgUmVjbGFpbSBwYWNrYWdlIGNhY2hlcyBiZWZvcmUgZXh0cmFjdGluZyBhc3NldHMgaW50byBLYWdnbGUncyBsaW1pdGVkIGRpc2suCmlmIFtbIC1uICIke0NPTkRBX0VYRTotfSIgJiYgLXggIiR7Q09OREFfRVhFfSIgXV07IHRoZW4KICAiJENPTkRBX0VYRSIgY2xlYW4gLS1hbGwgLS15ZXMgPi9kZXYvbnVsbCAyPiYxIHx8IHRydWUKICAiJENPTkRBX0VYRSIgcnVuIC1uIHdvcmszOCBweXRob24gLW0gcGlwIGNhY2hlIHB1cmdlID4vZGV2L251bGwgMj4mMSB8fCB0cnVlCmZpCgpkb3dubG9hZF9leHRyYWN0KCkgewogIGxvY2FsIHVybD0iJDEiIGFyY2hpdmU9IiQyIiBkZXN0aW5hdGlvbj0iJDMiCiAgbWtkaXIgLXAgIiRkZXN0aW5hdGlvbiIgIiQoZGlybmFtZSAiJGFyY2hpdmUiKSIKICBybSAtZiAiJGFyY2hpdmUiICIkYXJjaGl2ZS5wYXJ0IgogIHdnZXQgLXEgIiR1cmwiIC1PICIkYXJjaGl2ZS5wYXJ0IgogIG12ICIkYXJjaGl2ZS5wYXJ0IiAiJGFyY2hpdmUiCiAgdW56aXAgLXEgLW8gIiRhcmNoaXZlIiAtZCAiJGRlc3RpbmF0aW9uIgogIHJtIC1mICIkYXJjaGl2ZSIKfQoKQkFTRV9VUkw9Imh0dHBzOi8vZG93bmxvYWQuaXMudHVlLm1wZy5kZS9lbW9jYS9hc3NldHMiCmRvd25sb2FkX2V4dHJhY3QgIiRCQVNFX1VSTC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfbXNlXzIwLnppcCIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfbXNlXzIwLnppcCIgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscyIKZG93bmxvYWRfZXh0cmFjdCAiJEJBU0VfVVJML0VNT0NBL21vZGVscy9ERUNBLnppcCIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvREVDQS56aXAiICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMiCmRvd25sb2FkX2V4dHJhY3QgIiRCQVNFX1VSTC9ERUNBLnppcCIgIiRBU1NFVF9ST09UL0RFQ0EuemlwIiAiJEFTU0VUX1JPT1QiCmRvd25sb2FkX2V4dHJhY3QgIiRCQVNFX1VSTC9GYWNlUmVjb2duaXRpb24uemlwIiAiJEFTU0VUX1JPT1QvRmFjZVJlY29nbml0aW9uLnppcCIgIiRBU1NFVF9ST09UIgpkb3dubG9hZF9leHRyYWN0ICIkQkFTRV9VUkwvRkxBTUUuemlwIiAiJEFTU0VUX1JPT1QvRkxBTUUuemlwIiAiJEFTU0VUX1JPT1QiCgpmb3IgcmVxdWlyZWQgaW4gXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfbXNlXzIwIiBcCiAgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscy9ERUNBIiBcCiAgIiRBU1NFVF9ST09UL0RFQ0EiIFwKICAiJEFTU0VUX1JPT1QvRmFjZVJlY29nbml0aW9uIiBcCiAgIiRBU1NFVF9ST09UL0ZMQU1FIjsgZG8KICB0ZXN0IC1kICIkcmVxdWlyZWQiIHx8IHsgZWNobyAiUmVxdWlyZWQgRU1PQ0EgYXNzZXQgd2FzIG5vdCBleHRyYWN0ZWQ6ICRyZXF1aXJlZCIgPiYyOyBleGl0IDM7IH0KZG9uZQoKZWNobyAiUmVxdWlyZWQgRU1PQ0EgYXNzZXRzIHJlYWR5OyB1bnVzZWQgY2hlY2twb2ludHMgYW5kIGRlbW8gZGF0YSBza2lwcGVkLiIK'))
run_quiet(['bash', str(asset_setup), str(WORKDIR)], env=asset_env,
          label='EMOCA/FLAME asset download')


In [ ]:
# Smoke: two images at every angle, roll-only mode. Stale smoke rows are removed first.
RESULTS = WORKDIR / 'results' / 'roll_sweep'
SMOKE = RESULTS / 'smoke'
ANGLES_CSV = ','.join(str(angle) for angle in ANGLES)
THRESHOLDS_CSV = ','.join(str(t) for t in ROLL_THRESHOLDS)
ROLL_ARGS = ['--angles', ANGLES_CSV, '--mode', 'roll2d', '--min-roll', str(MIN_ROLL)]
shutil.rmtree(SMOKE, ignore_errors=True)
run_quiet([sys.executable, str(FRONTALIZE_SCRIPTS / 'launch_frontalize_workers.py'),
           '--bundle-root', str(BUNDLE_ROOT), '--workdir', str(WORKDIR),
           '--output-dir', str(SMOKE), '--limit', '2', *ROLL_ARGS],
          label='Roll sweep smoke inference')
rows = [json.loads(line) for shard in sorted(SMOKE.glob('shard_*.jsonl'))
        for line in shard.read_text(encoding='utf-8').splitlines()]
valid = [r for r in rows if any('label' in (r.get('raw') or {}).get(m, {}) for m in ('poster_rafdb', 'ma3d_rafdb'))]
if not valid:
    for r in rows[:3]:
        print(r.get('image'), r.get('angle'), {k: r[k] for k in r if k.endswith('_error')}, r.get('raw'))
    raise RuntimeError('Roll sweep smoke produced no valid predictions; full run not started.')
subprocess.run([sys.executable, str(FRONTALIZE_SCRIPTS / 'summarize_roll_sweep.py'),
                '--results-dir', str(SMOKE), '--thresholds', THRESHOLDS_CSV], check=True)
print(f'Roll sweep smoke passed ({len(valid)}/{len(rows)} rows with predictions).')


In [ ]:
# Full run: 3,068 images x len(ANGLES). Resumable: re-running this cell continues where it stopped.
FULL = RESULTS / 'full'
if RUN_FULL_BENCHMARK:
    run_quiet([sys.executable, str(FRONTALIZE_SCRIPTS / 'launch_frontalize_workers.py'),
               '--bundle-root', str(BUNDLE_ROOT), '--workdir', str(WORKDIR),
               '--output-dir', str(FULL), *ROLL_ARGS],
              label='Full RAF-DB roll sweep')
    subprocess.run([sys.executable, str(FRONTALIZE_SCRIPTS / 'summarize_roll_sweep.py'),
                    '--results-dir', str(FULL), '--thresholds', THRESHOLDS_CSV], check=True)


In [ ]:
# Zip the numeric results (no EMOCA logs) for download from the Output panel.
OUT = WORKDIR / 'roll_results'
shutil.rmtree(OUT, ignore_errors=True)
shutil.copytree(RESULTS / 'full', OUT / 'roll_sweep', ignore=shutil.ignore_patterns('emoca_gpu*.log'))
print('Download:', shutil.make_archive('/kaggle/working/roll_results', 'zip', OUT))
